# Preprocessing

In [1]:
import os
import json
from pprint import pprint
import torch
import ast
import pandas as pd
import concurrent.futures
import logging
os.environ["TQDM_DISABLE"] = "1" 
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"   # or "WARNING"
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"       # some versions use this
logging.getLogger("vllm").setLevel(logging.ERROR)
# device = torch.device("cuda:1")
print("CUDA available:", torch.cuda.is_available())
print("Number of CUDA devices:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Current CUDA device:", torch.cuda.current_device())
    print("CUDA device name:", torch.cuda.get_device_name(torch.cuda.current_device()))

CUDA available: True
Number of CUDA devices: 2
Current CUDA device: 0
CUDA device name: NVIDIA RTX A5000


In [2]:
!hf auth whoami



Not logged in


In [3]:
# read csv file
import pandas as pd
df = pd.read_csv('o4mini_annotation_final.csv')
df.head()


,vaers_id,symptom_text,symptom_list,llm_result,llm_result_formatted,t_result,t_corrected_version,t_category,t_notes,h_result,...,llm_errors,num_groups,agreement,text_length,llm_result_final,issues,llm_a1_agreement,llm_a2_agreement,a1_a2_agreement,llm_final_agreement
0,1420769,"About 4-5 days after my second dose, I began h...","['Chest discomfort', 'Chest pain', 'Electrocar...","[ \n { \n ""Chest pain"": [""non stop chest...","[\n {\n ""Chest pain"": [\n ""non stop c...",1,NaN,NaN,NaN,1,...,NaN,4.0,TRUE,110,"[\n {\n ""Chest pain"": [\n ""non stop c...",[],TRUE,TRUE,TRUE,TRUE
1,1527339,"cough, nasal congestion, runny nose, cold swea...","['Cold sweat', 'Cough', 'Fatigue', 'Hypogeusia...","""No temporal information found""",No temporal information found,1,NaN,NaN,NaN,1,...,NaN,NaN,TRUE,18,No temporal information found,[],TRUE,TRUE,TRUE,TRUE
2,1831772,"Muscle pain, headache, nausea. Muscle pain ins...","['Headache', 'Immediate post-injection reactio...","[\n {\n ""Headache"": [""none""],\n ""Immedi...","[\n {\n ""Myalgia"": [\n ""Muscle pain""\...",0,"[\n {\n ""Myalgia"": [\n ""Muscle pain""\...",Symptoms with no original mention should be gr...,NaN,0,...,NaN,2.0,FALSE,13,"[\n {\n ""Myalgia"": [\n ""Muscle pain""\...","['llm formatting issue', 'incorrect grouping',...",FALSE,FALSE,TRUE,FALSE
3,1021935,back pain/soreness in his lower back/had bad l...,"['Arthralgia', 'Back pain', 'Nausea', 'Pain in...","[\n {\n ""Vaccination site pain"": [""injecti...","[\n {\n ""Vaccination site pain"": [\n ...",1,NaN,NaN,NaN,0,...,NaN,2.0,0,260,"[\n {\n ""Vaccination site pain"": [\n ...",[],TRUE,0,0,TRUE
4,2749020,The above vaccines administered without incide...,"['Dizziness', 'Fall', 'Fatigue', 'Head injury'...","[\n {\n ""Syncope"": [""patient fainted""],\n ...","[\n {\n ""Syncope"": [\n ""patient faint...",1,NaN,NaN,NaN,1,...,NaN,2.0,1,118,"[\n {\n ""Syncope"": [\n ""patient faint...",[],1,1,1,1


In [4]:
df.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 27 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   vaers_id                 6000 non-null   int64  
 1   symptom_text             6000 non-null   object 
 2   symptom_list             6000 non-null   object 
 3   llm_result               6000 non-null   object 
 4   llm_result_formatted     6000 non-null   object 
 5   t_result                 6000 non-null   int64  
 6   t_corrected_version      814 non-null    object 
 7   t_category               1337 non-null   object 
 8   t_notes                  440 non-null    object 
 9   h_result                 6000 non-null   int64  
 10  h_corrected_version      188 non-null    object 
 11  h_notes                  615 non-null    object 
 12  final_result             6000 non-null   int64  
 13  final_corrected_version  896 non-null    object 
 14  final_notes             

In [5]:
# print out all the columns for one row from the dataframe
row = df.iloc[1089]
for col in df.columns:
    print(f"{col}: {row[col]}") 

vaers_id: 2599958
symptom_text: Narrative:  75 y/o male with PMH of T2DM, COPD, HTN, bilateral lower extremity DVT (in 01/2020), OSA (on CPAP), and gout presented the ED on 08/16/2021 for a 2 week history of worsening SOB, cough (with clear sputum), and weakness. In the ED, he met SIRS criteria with a temperature of 103.5 and a HR of 105. CXR demonstrated a left basilar airspace consolidation, so patient was given 1 L of NS and was ordered empiric vancomycin (not administered) and cefepime as part of the sepsis protocol. COVID test in the ED was positive (he is 2/2 for vaccinations), so pt was admitted for treatment/evaluation of presumptive COVID pneumonia with possible concurrent bacterial pneumonia (08/17/2021 procalcitonin 0.940). While admitted, patient always remained on room air. On 08/16/2021, he would transiently become hypoxic during conversation but this did not persist after treatment was initiated (i.e. after treatment initiation, SpO2 was usually steady in the low-to-mid 

In [6]:
# get columns from df as new df
data = df[['vaers_id','symptom_text', 'symptom_list','llm_result','llm_result_formatted','final_corrected_version','final_result']]
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 6000 non-null   int64 
 1   symptom_text             6000 non-null   object
 2   symptom_list             6000 non-null   object
 3   llm_result               6000 non-null   object
 4   llm_result_formatted     6000 non-null   object
 5   final_corrected_version  896 non-null    object
 6   final_result             6000 non-null   int64 
dtypes: int64(2), object(5)
memory usage: 328.2+ KB


In [7]:
# Gold timeline = human-corrected version when one exists, otherwise the accepted
# o4-mini result. A report has temporal progression unless that gold timeline is
# "No temporal information found". (The earlier filter also tested
# llm_result_formatted on its own, which dropped the 10 reports where o4-mini saw
# no progression but the annotators corrected it to a multi-stage timeline,
# giving 3156 instead of the 3166 MedTempo-T reports stated in the paper.)
data['gold_timeline'] = data['final_corrected_version'].where(
    data['final_corrected_version'].notna(), data['llm_result_formatted'])
data['no_temporal_info'] = data['gold_timeline'].astype(str).str.lower().str.contains('no temporal information')
data['no_temporal_info'].value_counts()
# get rows where no_temporal_info is False as new df
data_with_temporal_info = data[data['no_temporal_info'] == False]
data_with_temporal_info.info()
# we have to remove final_result value 2 from data_with_temporal_info
data_with_temporal_info = data_with_temporal_info[data_with_temporal_info['final_result'] != 2]
data_with_temporal_info.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3794 entries, 0 to 5998
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 3794 non-null   int64 
 1   symptom_text             3794 non-null   object
 2   symptom_list             3794 non-null   object
 3   llm_result               3794 non-null   object
 4   llm_result_formatted     3794 non-null   object
 5   final_corrected_version  654 non-null    object
 6   final_result             3794 non-null   int64 
 7   no_temporal_info         3794 non-null   bool  
dtypes: bool(1), int64(2), object(5)
memory usage: 240.8+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 3156 entries, 0 to 5995
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 3156 non-null   int64 
 1   symptom_text             3156 non-null   ob

/tmp/ipykernel_31235/3745764098.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['no_temporal_info'] = data.apply(lambda row:


In [8]:
# find data with no temporal info
data_no_temporal_info = data[data['no_temporal_info'] == True]
data_no_temporal_info = data_no_temporal_info[data_no_temporal_info['final_result'] != 2]

data_no_temporal_info.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2191 entries, 1 to 5999
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 2191 non-null   int64 
 1   symptom_text             2191 non-null   object
 2   symptom_list             2191 non-null   object
 3   llm_result               2191 non-null   object
 4   llm_result_formatted     2191 non-null   object
 5   final_corrected_version  238 non-null    object
 6   final_result             2191 non-null   int64 
 7   no_temporal_info         2191 non-null   bool  
dtypes: bool(1), int64(2), object(5)
memory usage: 139.1+ KB


In [9]:
# take 20 each from data_with_temporal_info and data_no_temporal_info
data_sampled_with_temporal_info = data_with_temporal_info
data_sampled_no_temporal_info = data_no_temporal_info
# concatenate the two dataframes
data_sampled = pd.concat([data_sampled_with_temporal_info, data_sampled_no_temporal_info])
data_sampled.info()
# save to csv
# data_sampled.to_csv('sample_40.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
Index: 5347 entries, 0 to 5999
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 5347 non-null   int64 
 1   symptom_text             5347 non-null   object
 2   symptom_list             5347 non-null   object
 3   llm_result               5347 non-null   object
 4   llm_result_formatted     5347 non-null   object
 5   final_corrected_version  889 non-null    object
 6   final_result             5347 non-null   int64 
 7   no_temporal_info         5347 non-null   bool  
dtypes: bool(1), int64(2), object(5)
memory usage: 339.4+ KB


In [10]:
# # make some sample that we can use to test
# data_sampled_with_temporal_info.to_csv('sample_our_method_with_temporal_info_200.csv', index=False)
# data_sampled_no_temporal_info.to_csv('sample_our_method_no_temporal_info_100.csv', index=False)
# data_sampled.to_csv('sample_our_method_total_300.csv', index=False)

In [11]:
# now lets take 200 random samples from data_with_temporal_info with random_state=2
sample = data_sampled_with_temporal_info
sample.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3156 entries, 0 to 5995
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 3156 non-null   int64 
 1   symptom_text             3156 non-null   object
 2   symptom_list             3156 non-null   object
 3   llm_result               3156 non-null   object
 4   llm_result_formatted     3156 non-null   object
 5   final_corrected_version  651 non-null    object
 6   final_result             3156 non-null   int64 
 7   no_temporal_info         3156 non-null   bool  
dtypes: bool(1), int64(2), object(5)
memory usage: 200.3+ KB


In [12]:
# # split the data in halves according to the length of the data and run the first half
# first_half, second_half = sample.iloc[:len(sample)//2], sample.iloc[len(sample)//2:]
# sample = first_half
# sample.info()
# sample.head()

# Prompt body

In [13]:
from string import Template
import re
temporal_id_prompt_tpl = Template(r"""
You are an expert medical analyst. Your task is to determine if there is a temporal progression of different adverse events in the clinical notes. 

Definition: Temporal progression means a sequence where a symptom not previously mentioned appears after a previously mentioned symptom. The new symptoms must be distinct from any symptoms already listed in the clinical notes. 

What does not constitute temporal progression: 
- Concurrent Onset: All symptoms begin at the same time. 
- Symptom Resolution: A symptom disappears or improves. 
- Fluctuating Severity: A symptom gets better and worse. 
- Persistence: A symptom persists. 
- Inferred Timelines: Timelines based on symptom durations alone. For example, "Headache for 4 days, fever for 3 days" does not demonstrate temporal progression. 
- Assumptions: Any assumptions about the order of events if not directly stated in the text. 
- Re-emergence of Previous Symptoms: Re-emergence or continuation of previously reported symptoms, even with changes in severity, does not constitute temporal progression. The sequence must involve entirely new symptoms. 

Focus: Focus on whether different adverse events appear at different times, as explicitly stated in the clinical notes. Do not consider changes to the same adverse event over time. Pay close attention to whether the clinical notes introduce a symptom that has not been mentioned previously. 

Example of what does not constitute temporal progression: "Patient reported fatigue for 5 days and a cough for 2 days, then the fatigue returned." The return of fatigue is not a new event. Another example: "Patient reported headache and nausea, then the headache persisted the next day. This is not temporal progression." 

Clinical Notes:
$note

Output:
- If a temporal progression of different adverse events is identified, return exactly: Temporal information found
- If no temporal progression of different adverse events is found, return exactly: No temporal information found
""").substitute
# Keep your existing heuristic
_TEMPORAL_MARKERS_RE = re.compile(
    r"\b(then|after|later|subsequent(?:ly)?|the\s+next\s+day|the\s+following\s+day|day\s*\d+|week\s*\d+|post[-\s]|prior\s+to|initially|thereafter|by\s+day|on\s+day|on\s+\d{1,2}/\d{1,2}/\d{2,4})\b",
    flags=re.IGNORECASE
)

In [14]:
# first version of prompt template

from string import Template

generation_prompt =  Template(""" 
Ignore previous conversations. 
 
TASK: 
Temporally order the provided list of adverse events based on their sequence of appearance in the clinical notes and extract the most specific mention for each adverse event from the notes. 
 
Clinical Notes: $symptom_text

 
Adverse Events to Temporally Order: $symptom_list
 
 
PROCESSING INSTRUCTIONS: 
- Start with the provided adverse events list and reorder it based on the sequence implied in the clinical notes. 
- If several adverse effects are mentioned together without a clear temporal order, group them into the same block (inside a single dictionary). 
- For each adverse event, extract the closest matching phrase from the clinical notes; if an adverse event is not mentioned, assign "none" as its value. 
 
IMPORTANT RULES: 
- No Invention: Never add, remove, or modify symptom names from the provided list. 
- Mention Each Symptom Once: Mention each symptom only once — at the first time it appears or becomes relevant. 
- Specific vs Generic Terms: When a phrase matches both specific and generic symptoms, assign it to the specific term only. If "Lip swelling" or "Pharyngeal swelling" is matched, do NOT include the generic "Swelling" unless there is a separate, explicit mention of general swelling elsewhere. 
- Temporal Evidence Required: If multiple symptoms are mentioned across multiple sentences without clear timeline separation, group them together. Do NOT treat different sentences as different times unless there is an explicit temporal indicator (e.g., "then", "after that", "later", specific dates or times. 
- Unmentioned Symptoms: Group all symptoms with no original mention together in a separate block with "none" as their value. This block should be placed after all the temporally ordered groups. 
 
OUTPUT FORMAT: 
Return your answer only in valid JSON format — using a list of dictionaries to represent temporal progression and grouping, as shown below: 
[
  {"Erythema": ["redness in neck"],
  {
    "Pain in extremity": ["sore arm"],
    "Pruritus": ["itchy feeling"]
  },
  {"Swelling": ["mild arm swelling"]
]
                                 
# When revising, consider both the feedback (if any) and the previous attempt result (if provided).
# Keep correct parts from prior attempts, but fix issues based on feedback.

$prev_result_block
$feedback_block
""").substitute



In [15]:
# --------- Prompt templates ----------

verify_prompt_tpl = Template("""
Given the original text and extracted symptoms below:

Original Text:
$symptom_text

Symptoms to Extract:
$symptom_list

Current Result (JSON):
$initial_result

Scoring rubric (0–5), +1 each if:
1) The JSON is valid and groups are earliest→latest,
2) **Non-mentioned symptoms** should be presented as "none" in the **last** group,
3) Every symptom in the list appears exactly once overall,
4) Symptoms grouped together occur around the same time,
5) Group ordering follows the text's temporal cues.

Return ONLY one of the following JSON objects:
- If score >= 3: {"score": <int>, "feedback": ""}   # feedback can be empty string
- If score < 3:  {"score": <int>, "feedback": "<specific, actionable fixes>"}
""").substitute

In [16]:
# modified two prompt structure
generation_prompt_init = Template("""
Ignore previous conversations.

TASK:
Temporally order the provided list of adverse events based on their sequence of appearance in the clinical notes and extract the most specific mention for each adverse event from the notes.

Clinical Notes: $symptom_text

Adverse Events to Temporally Order: $symptom_list

PROCESSING INSTRUCTIONS:
- Start with the provided adverse events list and reorder it based on the sequence implied in the clinical notes.
- If several adverse effects are mentioned together without a clear temporal order, group them into the same block (inside a single dictionary).
- For each adverse event, extract the closest matching phrase from the clinical notes. The matching phrase MUST be a verbatim, contiguous quote copied from the clinical notes (exact substring); 
if an adverse event is not mentioned, assign "none" as its value.

IMPORTANT RULES:
- No Invention: Never add, remove, or modify symptom names from the provided list.
- Mention Each Symptom Once: Mention each symptom only once — at the first time it appears or becomes relevant.
- Specific vs Generic Terms: When a phrase matches both specific and generic symptoms, assign it to the specific term only. If "Lip swelling" or "Pharyngeal swelling" is matched, do NOT include the generic "Swelling" unless there is a separate, explicit mention of general swelling elsewhere.
- Temporal Evidence Required: If multiple symptoms are mentioned across multiple sentences without clear timeline separation, group them together. Do NOT treat different sentences as different times unless there is an explicit temporal indicator (e.g., "then", "after that", "later", specific dates or times).
- Unmentioned Symptoms: Group all symptoms with no original mention together in a separate block with "none" as their value. This block should be placed after all the temporally ordered groups.

OUTPUT FORMAT:
Return your answer only in valid JSON format — using a list of dictionaries to represent temporal progression and grouping, as shown below:
{[
  {"Erythema": ["redness in neck"]},
  {
    "Pain in extremity": ["sore arm"],
    "Pruritus": ["itchy feeling"]
  },
  {"Swelling": ["mild arm swelling"]}
]}

""").substitute

generation_prompt_edit = Template("""
Ignore previous conversations.

Clinical Notes: $symptom_text

Adverse Events List (canonical keys; do not add/remove/rename keys): $symptom_list

OUTPUT FORMAT (must follow exactly):
Return only valid JSON: a list of dictionaries representing temporal progression.
All keys must come from the provided adverse events list.
For each key symptom, the value MUST be a verbatim, contiguous quote copied from the clinical notes (exact substring). If you cannot find an exact substring, output "none" for that symptom.
Group all unmentioned symptoms with value "none" into a final block placed last.


Previous result JSON:
$prev_result_block

Verifier feedback:
$feedback_block

Edit Executor Requirement (must follow exactly):
If the verifier feedback contains operations in the form "OP=...", you MUST apply those operations literally to the CURRENT candidate timeline JSON and return the updated timeline.
- Do NOT re-decide the entire timeline.
- Do NOT change any group order unless an OP explicitly requests SWAP or MERGE.
- Do NOT move any symptom key unless explicitly listed in a MOVE or SPLIT operation.
- Do NOT introduce new timepoints unless explicitly requested by SPLIT (NEW_AFTER_i).
- Keep the output schema identical: a list of timepoint dicts with canonical keys mapping to "none" or extracted text/list.
After applying the operations, only revise the minimal necessary content to keep the timeline coherent; do not add new symptoms or remove unrelated ones.

""").substitute

In [17]:
verify_prompt_anchor_tpl = Template("""
Ignore previous conversations.

You are a temporal consistency verifier for a candidate adverse-event timeline.

Clinical Notes:
$NOTE_TEXT

Adverse Events List (canonical keys):
$SYMPTOM_LIST

Candidate Timeline JSON (a list of timepoints; each timepoint is a dict from canonical key -> "none" or extracted text/list):
$CANDIDATE_JSON

Goal:
Evaluate ONLY the temporal sequence quality (ordering + grouping). Ignore whether extracted mention strings are exact quotes.

ANCHOR-ONLY verification method (do NOT output anchors or reasoning):
1) Infer a dominant time anchor Ai for each timepoint i using explicit dates/day counts and strong temporal phrases in the note.
   - Ai can be an explicit date/day (e.g., 3/6, March 8, Day 2, post-vax day 1), a clear stage boundary (e.g., ED visit/admitted/discharged/returned/follow-up), or Unknown.
2) Chronology:
   - Penalize only CLEAR contradictions where the order of timepoints conflicts with explicit anchor order in the note.
3) Grouping validity (using anchors):
   - Separation (over-merge): If a single timepoint clearly mixes events tied to different anchors, it is invalid.
   - Cohesion (over-split): If two adjacent timepoints clearly share the same anchor and there is no clear stage/transition boundary, the split is likely unnecessary.
4) Minimality/support:
   - Prefer the simplest timeline consistent with anchors (avoid extra timepoints without distinct anchor justification).

Scoring (universal 0–5, strict and conservative):
Start at 5. Subtract 1 for each CLEAR violation (max 5):
- Clear chronology contradiction with explicit anchors
- Clear over-merge: mixed anchors within one timepoint
- Clear over-split: redundant adjacent timepoints without distinct anchor/stage justification
- Clear lack of temporal support: too many timepoints despite only one anchor/stage in the note
- Overall timeline clearly inconsistent with temporal cues in the note

Conservatism rule:
If the note is ambiguous and you cannot identify a CLEAR violation, do NOT subtract points. Do NOT guess.

Feedback requirement (only when score < 4):
Return an actionable edit plan with AT MOST 2 operations, using ONLY the following operations:
- SWAP(g_i,g_j)
- MERGE(g_i,g_{i+1})
- MOVE(<symptom_key>, g_from, g_to)
- SPLIT(g_i, move_keys=[<symptom_key>, ...], to=NEW_AFTER_i)

Guidelines for feedback:
- Use 0-based group indices (g0, g1, ...).
- Refer to events ONLY by canonical symptom keys (not free-text).
- Prefer minimal edits (1–2 ops) that fix the most severe issue first.
- If you cannot propose a confident edit, set score to 3 and explain briefly what is ambiguous.

Output ONLY valid JSON:
- If score >= 3: {"score": <int>, "feedback": ""}
- If score < 3:  {"score": <int>, "feedback": "ASPECT=<CHRONOLOGY|GROUPING|MINIMALITY>; OP=<...>; OP=<...>"}
Do not output anchors, quotes, or any reasoning.

""").substitute

# Modeling

In [18]:
# ============================
# MODELING SECTION (vLLM + batching)
# Paste this as a single cell.
# Assumes you already have:
#   - generation_prompt / generation_prompt_init / gen_prompt (your generator prompt template)
#   - verify_prompt_tpl (your verifier prompt template)
#   - pred_source DataFrame with columns: ["vaers_id", "symptom_text", "symptom_list"]
# Evaluation code remains unchanged.
# ============================

import os, json, re, ast, time
from typing import Any, Optional, Dict, List, Tuple
import pandas as pd
import torch

from vllm import LLM, SamplingParams

# ----------------------------
# Config
# ----------------------------
# Make sure CUDA sees BOTH GPUs (optional if your env already does)
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

MODEL_ID = "shuyuej/Llama-3.3-70B-Instruct-GPTQ"  # change if needed
HF_TOKEN = None  # set if needed for gated models

TENSOR_PARALLEL_SIZE = 2
GPU_MEMORY_UTILIZATION = 0.92
MAX_MODEL_LEN = 8192    #16384   

# Iteration & batching knobs
MAX_ITERATIONS = 4
MICRO_BATCH_SIZE = 2        # start with 2 if OOM; try 4 if stable
GEN_MAX_TOKENS = 256        # keep your existing; verifier should be larger
VER_MAX_TOKENS = 512        # ensure verifier has the most max_new_tokens (>= GEN_MAX_TOKENS)
GEN_TEMPERATURE = 0.0
VER_TEMPERATURE = 0.0
ACCEPT_SCORE = 3

# Optional: temporal gate (only if your temporal_id_prompt_tpl exists in notebook)
USE_TEMPORAL_GATE = False 
TEMPORAL_CONF_THRESHOLD = 0.5
QUANTIZATION = "gptq_marlin"     # quantization

# ----------------------------
# vLLM init (lazy)
# ----------------------------
_vllm_llm: Optional[LLM] = None
_vllm_tokenizer = None

# =========================
# PROMPT COMBO SWITCHBOARD
# (only edit these between runs)
# =========================
GEN_PROMPT_MODE = "init_edit"   # "traditional" OR "init_edit"
VER_PROMPT_MODE = "anchor"   # "traditional" OR "anchor"

def reset_model():
    """Call this after changing MODEL_ID / MAX_MODEL_LEN / TP settings."""
    global _vllm_llm, _vllm_tokenizer
    _vllm_llm = None
    _vllm_tokenizer = None
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def _load_vllm_once():
    global _vllm_llm, _vllm_tokenizer
    if _vllm_llm is not None and _vllm_tokenizer is not None:
        return

    if not torch.cuda.is_available():
        raise RuntimeError("vLLM GPU inference expected (you are loading a 70B model).")

    if HF_TOKEN and not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = HF_TOKEN

    dtype_str = "bfloat16" if (hasattr(torch.cuda, "is_bf16_supported") and torch.cuda.is_bf16_supported()) else "float16"
    # AWQ requires fp16 in vLLM
    if QUANTIZATION == "awq" or QUANTIZATION == "gptq_marlin":
        dtype_str = "float16"
    print(f"[vLLM] Loading {MODEL_ID} | tp={TENSOR_PARALLEL_SIZE} | dtype={dtype_str} | max_model_len={MAX_MODEL_LEN}")

    # For pre-quantized *-bnb-4bit repos, vLLM will infer quantization from model config.
    _vllm_llm = LLM(
        model=MODEL_ID,
        tensor_parallel_size=TENSOR_PARALLEL_SIZE,
        dtype=dtype_str,
        quantization=QUANTIZATION,     # quantization
        trust_remote_code=True,
        max_model_len=MAX_MODEL_LEN,
        gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
        disable_log_stats=True,  
        distributed_executor_backend="mp", # gptq
        disable_custom_all_reduce=True,
        enforce_eager=True,  # disables torch.compile/inductor path that is hitting Python.h
    )
    _vllm_tokenizer = _vllm_llm.get_tokenizer()

    # align pad token to eos if missing
    if getattr(_vllm_tokenizer, "pad_token_id", None) is None and getattr(_vllm_tokenizer, "eos_token_id", None) is not None:
        _vllm_tokenizer.pad_token = _vllm_tokenizer.eos_token


# ----------------------------
# Prompt rendering helpers (Template / .substitute / format / callable)
# ----------------------------
def _tpl_src(tpl) -> str:
    """Best-effort access to the underlying template text (for placeholder detection)."""
    if tpl is None:
        return ""
    if isinstance(tpl, str):
        return tpl
    if hasattr(tpl, "template") and isinstance(getattr(tpl, "template"), str):
        return tpl.template
    # Bound method like Template(...).substitute
    if hasattr(tpl, "__self__") and hasattr(tpl.__self__, "template") and isinstance(getattr(tpl.__self__, "template"), str):
        return tpl.__self__.template
    return ""


def _tpl_has(tpl, name: str) -> bool:
    s = _tpl_src(tpl)
    return (f"${name}" in s) or (f"{{{name}}}" in s)


def _render(tpl, **kwargs) -> str:
    """Render a prompt template.

    Supports:
      - str with `{name}` placeholders
      - string.Template / its `.substitute` bound method with `$name` placeholders
      - callables that accept **kwargs

    Always falls back to safe_substitute to avoid KeyError explosions.
    """
    from string import Template

    if tpl is None:
        raise ValueError("Prompt template is None. Define generation_prompt(_init/_edit) and verify_prompt(_anchor)_tpl.")

    # 1) Plain string: try .format(), then safe_substitute.
    if isinstance(tpl, str):
        try:
            return tpl.format(**kwargs)
        except Exception:
            return Template(tpl).safe_substitute(kwargs)

    # 2) string.Template object: try its raw .template with .format first (for {VAR} style).
    if hasattr(tpl, "template") and isinstance(getattr(tpl, "template"), str):
        s = tpl.template
        try:
            return s.format(**kwargs)
        except Exception:
            pass
        # Then try $VAR substitution.
        if hasattr(tpl, "substitute"):
            try:
                return tpl.substitute(kwargs)
            except Exception:
                try:
                    return tpl.safe_substitute(kwargs)
                except Exception:
                    pass
        return Template(s).safe_substitute(kwargs)

    # 3) Callable: try call; if it looks like Template.substitute bound method, downgrade to safe_substitute.
    if callable(tpl):
        try:
            return tpl(**kwargs)
        except Exception:
            if hasattr(tpl, "__self__") and hasattr(tpl.__self__, "safe_substitute"):
                try:
                    return tpl.__self__.safe_substitute(kwargs)
                except Exception:
                    pass

    # Last resort.
    return Template(str(tpl)).safe_substitute(kwargs)
def _messages_to_prompt(messages: List[Dict[str, str]]) -> str:
    """Use model chat template if available; otherwise fallback to a simple transcript."""
    _load_vllm_once()
    tok = _vllm_tokenizer

    if hasattr(tok, "apply_chat_template"):
        try:
            return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            pass

    # Fallback transcript (works, but less ideal than true chat template)
    chunks = []
    for m in messages:
        role = (m.get("role") or "user").strip().upper()
        content = m.get("content") or ""
        chunks.append(f"{role}:\n{content}\n")
    chunks.append("ASSISTANT:\n")
    return "\n".join(chunks)

def _get_eos_ids() -> Optional[List[int]]:
    """Try common end-of-turn tokens to stop earlier (saves time)."""
    _load_vllm_once()
    tok = _vllm_tokenizer
    eos_ids: List[int] = []
    if getattr(tok, "eos_token_id", None) is not None:
        eos_ids.append(tok.eos_token_id)

    for t in ["<|eot_id|>", "<|end_of_turn|>", "<|im_end|>", "<|endoftext|>", "</s>"]:
        try:
            tid = tok.convert_tokens_to_ids(t)
            if isinstance(tid, int) and tid != -1:
                unk = getattr(tok, "unk_token_id", None)
                if unk is None or tid != unk:
                    if tid not in eos_ids:
                        eos_ids.append(tid)
        except Exception:
            pass

    return eos_ids or None


# ----------------------------
# JSON helpers
# ----------------------------
def parse_json_best_effort(text: str) -> Optional[Any]:
    """Parse JSON from LLM output, tolerating fenced code blocks or leading text."""
    if not text:
        return None
    s = text.strip()
    # direct
    try:
        return json.loads(s)
    except Exception:
        pass
    # fenced ```json ... ```
    m = re.search(r"```(?:json)?\s*(\{.*?\}|\[.*?\])\s*```", s, flags=re.S)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass
    # first JSON-looking slice
    start = min([p for p in (s.find("{"), s.find("[")) if p != -1], default=-1)
    if start != -1:
        for end in range(len(s) - 1, start, -1):
            if s[end] in "}]":
                chunk = s[start:end + 1]
                try:
                    return json.loads(chunk)
                except Exception:
                    continue
    return None

def _parse_json_lenient(x: Any) -> Optional[Any]:
    if isinstance(x, (dict, list)):
        return x
    if not isinstance(x, str):
        return None
    s = x.strip().strip('"')
    if s == "No temporal information found":
        return s
    if s.startswith("```json") and s.endswith("```"):
        s = s[len("```json"):-3].strip()
    try:
        return json.loads(s)
    except Exception:
        pass
    m = re.search(r"```(?:json)?\s*(\{.*?\}|\[.*?\])\s*```", s, flags=re.S)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass
    start = min([p for p in (s.find("{"), s.find("[")) if p != -1], default=-1)
    if start != -1:
        for end in range(len(s) - 1, start, -1):
            if s[end] in "}]":
                chunk = s[start:end + 1]
                try:
                    return json.loads(chunk)
                except Exception:
                    continue
    return None


def normalize_symptom_list(symptom_list: Any) -> List[str]:
    """
    Accept list/tuple/set, JSON string list, or Python-list string (single quotes).
    Returns clean, deduplicated List[str].
    """
    if isinstance(symptom_list, (list, tuple, set)):
        seq = list(symptom_list)
    elif isinstance(symptom_list, str):
        s = symptom_list.strip()
        # JSON first
        try:
            maybe = json.loads(s)
            seq = list(maybe) if isinstance(maybe, (list, tuple, set)) else [maybe]
        except Exception:
            # Python literal next
            try:
                maybe = ast.literal_eval(s)
                seq = list(maybe) if isinstance(maybe, (list, tuple, set)) else [maybe]
            except Exception:
                # fallback: treat as one symptom
                seq = [s]
    else:
        seq = [str(symptom_list)]

    out, seen = [], set()
    for item in seq:
        t = "" if item is None else str(item).strip()
        if t and t not in seen:
            out.append(t)
            seen.add(t)
    return out


def format_check_and_fix(
    raw_or_obj: Any,
    symptom_list: List[str],
    *,
    accept_envelope_keys: Tuple[str, ...] = ("temporal_result", "result", "output"),
    generic_terms: Tuple[str, ...] = ("Condition aggravated",),
    none_tokens: Tuple[str, ...] = ("none",),
) -> Dict[str, Any]:
    """
    Canonicalizes output into List[Dict[str, "none" | List[str]]], earliest->latest,
    consolidates all "none" into last group, ensures full coverage for symptom_list,
    and DROPS keys not in symptom_list.
    """
    symptom_list = normalize_symptom_list(symptom_list)
    allowed = set(symptom_list)

    report_issues: List[str] = []
    changes = {
        "unwrapped_envelope": False,
        "coerced_non_list": 0,
        "removed_generic_terms": 0,
        "dropped_unknown_keys": 0,
        "moved_none_to_last": 0,
        "deduped_duplicates": 0,
        "added_missing_none": 0,
        "removed_empty_groups": 0,
        "normalized_values": 0,
    }

    obj = _parse_json_lenient(raw_or_obj)
    if obj is None:
        report_issues.append("parse_failed")
        fixed = [{sym: "none" for sym in symptom_list}]
        changes["added_missing_none"] = len(symptom_list)
        return {"fixed": fixed, "report": {"issues": report_issues, "changes": changes}}

    if obj == "No temporal information found":
        fixed = [{sym: "none" for sym in symptom_list}]
        changes["added_missing_none"] = len(symptom_list)
        return {"fixed": fixed, "report": {"issues": [], "changes": changes}}

    payload = obj
    if isinstance(obj, dict):
        for k in accept_envelope_keys:
            if k in obj:
                payload = obj[k]
                changes["unwrapped_envelope"] = True
                break

    if isinstance(payload, dict):
        payload = [payload]
        changes["coerced_non_list"] += 1
    if not isinstance(payload, list):
        report_issues.append("not_list_after_unwrap")
        payload = [{}]

    generic_set = set(generic_terms)
    none_set = set(t.lower() for t in none_tokens)

    norm_groups: List[Dict[str, Any]] = []
    for grp in payload:
        if not isinstance(grp, dict):
            changes["coerced_non_list"] += 1
            continue
        new_grp: Dict[str, Any] = {}

        for k, v in grp.items():
            # drop unknown keys
            if k not in allowed:
                if k in generic_set:
                    changes["removed_generic_terms"] += 1
                else:
                    changes["dropped_unknown_keys"] += 1
                continue

            # normalize value -> "none" OR non-empty List[str]
            val_is_none = False
            if v is None:
                val_is_none = True
            elif isinstance(v, str):
                val_is_none = v.strip().lower() in none_set
            elif isinstance(v, list):
                vv = [str(t).strip() for t in v if str(t).strip()]
                vv = [t for t in vv if t.lower() not in none_set]
                if len(vv) == 0:
                    val_is_none = True
                else:
                    v = vv
            else:
                vv = str(v).strip()
                if not vv:
                    val_is_none = True
                else:
                    v = [vv]

            if val_is_none:
                new_grp[k] = "none"
                changes["normalized_values"] += 1
            else:
                if isinstance(v, str):
                    v = [v.strip()] if v.strip() else []
                v = [t for t in v if t]
                if not v:
                    new_grp[k] = "none"
                    changes["normalized_values"] += 1
                else:
                    new_grp[k] = v

        if new_grp:
            norm_groups.append(new_grp)
        else:
            changes["removed_empty_groups"] += 1

    # keep earliest non-none per symptom
    first_non_none_idx: Dict[str, int] = {}
    for idx, grp in enumerate(norm_groups):
        for k, v in grp.items():
            if v != "none" and k not in first_non_none_idx:
                first_non_none_idx[k] = idx

    cleaned_groups: List[Dict[str, Any]] = []
    for i, grp in enumerate(norm_groups):
        out: Dict[str, Any] = {}
        for k, v in grp.items():
            if v == "none":
                changes["moved_none_to_last"] += 1
                continue
            if first_non_none_idx.get(k, i) == i:
                out[k] = v
            else:
                changes["deduped_duplicates"] += 1
        if out:
            cleaned_groups.append(out)

    present_syms = set().union(*(set(g.keys()) for g in cleaned_groups)) if cleaned_groups else set()
    missing = sorted(list(set(symptom_list) - present_syms))
    if missing:
        cleaned_groups.append({k: "none" for k in missing})
        changes["added_missing_none"] += len(missing)

    if not cleaned_groups:
        cleaned_groups = [{sym: "none" for sym in symptom_list}]
        changes["added_missing_none"] = len(symptom_list)

    return {"fixed": cleaned_groups, "report": {"issues": report_issues, "changes": changes}}


# ----------------------------
# vLLM batched call
# ----------------------------
def call_llm_batch(
    messages_batch: List[List[Dict[str, str]]],
    *,
    temperature: float,
    max_output_tokens: int,
    micro_batch_size: int,
) -> List[str]:
    _load_vllm_once()

    prompts = [_messages_to_prompt(msgs) for msgs in messages_batch]
    # n = len(_vllm_tokenizer(prompts[0]).input_ids)
    # print("prompt tokens:", n)

    
    do_sample = (temperature is not None and temperature > 0)
    sp = SamplingParams(
        max_tokens=max_output_tokens,
        temperature=(temperature if do_sample else 0.0),
        top_p=(0.95 if do_sample else 1.0),
        stop_token_ids=_get_eos_ids(),
    )

    outs_all: List[str] = []
    for i in range(0, len(prompts), micro_batch_size):
        chunk = prompts[i:i + micro_batch_size]
        outs = _vllm_llm.generate(chunk, sp, use_tqdm=False)

        # preserves order for list input
        for o in outs:
            txt = o.outputs[0].text if (o.outputs and len(o.outputs) > 0) else ""
            outs_all.append((txt or "").strip())

    return outs_all


# ----------------------------
# Optional: Temporal gate (batched)
# ----------------------------
def temporal_regex_heuristic(note_text: str) -> Dict[str, Any]:
    # cheap signals used only for logging / fallback
    signals = []
    # simple markers
    if re.search(r"\b(then|after that|later|next day|the next day|subsequently|afterwards)\b", note_text, flags=re.I):
        signals.append("temporal_marker_word")
    if re.search(r"\b(day\s*\d+|week\s*\d+|\d+\s*(days|weeks)\s*later)\b", note_text, flags=re.I):
        signals.append("relative_time")
    conf = 0.7 if signals else 0.2
    return {"has_progression": bool(signals), "signals": signals, "confidence": conf}

def temporal_id_batch(note_texts: List[str], *, min_confidence: float) -> List[Dict[str, Any]]:
    tpl = globals().get("temporal_id_prompt_tpl", None)
    if tpl is None:
        # prompt not defined in notebook -> disable gate
        return [{"has_progression": True, "signals": ["gate_disabled"], "confidence": 1.0} for _ in note_texts]

    msgs_batch = []
    for note in note_texts:
        # pass multiple placeholder names to be robust across your prompt variants
        user_prompt = _render(tpl, note=note, note_text=note, symptom_text=note)
        msgs_batch.append([
            {"role": "system", "content": "Return exactly one of: 'Temporal information found' or 'No temporal information found'. No extra words."},
            {"role": "user", "content": user_prompt},
        ])

    outs = call_llm_batch(
        msgs_batch,
        temperature=0.0,
        max_output_tokens=16,
        micro_batch_size=MICRO_BATCH_SIZE,
    )

    decisions: List[Dict[str, Any]] = []
    for note, raw in zip(note_texts, outs):
        raw_norm = (raw or "").strip().lower()
        heuristic = temporal_regex_heuristic(note)

        if "no temporal information found" in raw_norm:
            d = {"has_progression": False, "signals": heuristic["signals"], "confidence": 1.0}
        elif "temporal information found" in raw_norm:
            d = {"has_progression": True, "signals": heuristic["signals"], "confidence": 1.0}
        else:
            d = heuristic

        if d["confidence"] < min_confidence:
            d["has_progression"] = False
        decisions.append(d)

    return decisions


# ----------------------------
# Batch Agent (gen + format_fix + verify) with early-stop
# ----------------------------
class TemporalExtractionBatchAgent:
    def __init__(
        self,
        *,
        max_iterations: int = MAX_ITERATIONS,
        micro_batch_size: int = MICRO_BATCH_SIZE,
        gen_max_tokens: int = GEN_MAX_TOKENS,
        ver_max_tokens: int = VER_MAX_TOKENS,
        gen_temperature: float = GEN_TEMPERATURE,
        ver_temperature: float = VER_TEMPERATURE,
        accept_score: int = ACCEPT_SCORE,
        use_temporal_gate: bool = USE_TEMPORAL_GATE,
        temporal_conf_threshold: float = TEMPORAL_CONF_THRESHOLD,
    ):
        self.max_iterations = max_iterations
        self.micro_batch_size = micro_batch_size
        self.gen_max_tokens = gen_max_tokens
        self.ver_max_tokens = ver_max_tokens
        self.gen_temperature = gen_temperature
        self.ver_temperature = ver_temperature
        self.accept_score = accept_score
        self.use_temporal_gate = use_temporal_gate
        self.temporal_conf_threshold = temporal_conf_threshold

    def _build_gen_messages(
        self,
        symptom_text: str,
        symptom_list: List[str],
        prior_feedback: str,
        prior_result: Optional[Any],
    ) -> List[Dict[str, str]]:
        symptom_list_json = json.dumps(symptom_list, ensure_ascii=False, separators=(",", ":"))

        # Decide init vs edit prompt.
        is_edit_round = bool(prior_feedback) or (prior_result is not None and str(prior_result).strip() != "")

        if GEN_PROMPT_MODE == "traditional":
            gen_tpl = generation_prompt
        else:
            gen_tpl = generation_prompt_edit if is_edit_round else generation_prompt_init
        
        prev_result_block = ""
        if prior_result is not None and str(prior_result).strip() != "":
            prev_result_block = "Previous result:\n" + json.dumps(prior_result, ensure_ascii=False, indent=2)

        feedback_block = ""
        if prior_feedback:
            feedback_block = "Verifier feedback:\n" + str(prior_feedback).strip()

        # Render with both lowercase + uppercase aliases (supports your different prompt templates).
        user_prompt = _render(
            gen_tpl,
            symptom_text=symptom_text,
            symptom_list=symptom_list_json,
            prev_result_block=prev_result_block,
            feedback_block=feedback_block,
            # common alternates
            # NOTE_TEXT=symptom_text,
            # SYMPTOM_TEXT=symptom_text,
            # SYMPTOM_LIST=symptom_list_json,
            # PREV_RESULT_BLOCK=prev_result_block,
            # FEEDBACK_BLOCK=feedback_block,
        )

        # Backward-compat: if your template doesn't include these fields, append them.
        extra = []
        if prev_result_block and not (_tpl_has(gen_tpl, "prev_result_block") or _tpl_has(gen_tpl, "PREV_RESULT_BLOCK")):
            extra.append(prev_result_block)
        if feedback_block and not (_tpl_has(gen_tpl, "feedback_block") or _tpl_has(gen_tpl, "FEEDBACK_BLOCK")):
            extra.append(feedback_block)
        if extra:
            user_prompt = user_prompt.rstrip() + "\n\n" + "\n\n".join(extra)

        return [
            {"role": "system", "content": "You are a medical temporal sequence analysis assistant. Respond ONLY with a single JSON object in the specified shape."},
            {"role": "user", "content": user_prompt},
        ]
    def _build_ver_messages(
        self,
        symptom_text: str,
        symptom_list: List[str],
        fixed_list: List[Dict[str, Any]],
    ) -> List[Dict[str, str]]:
        symptom_list_json = json.dumps(symptom_list, ensure_ascii=False, separators=(",", ":"))
        wrapped_result_str = json.dumps({"temporal_result": fixed_list}, ensure_ascii=False, separators=(",", ":"))
        candidate_blocks_str = json.dumps(fixed_list, ensure_ascii=False, separators=(",", ":"))

        # normal_tpl = globals().get("ver_prompt", None) or globals().get("verify_prompt_tpl", None)
        # anchor_tpl = globals().get("verify_prompt_anchor_tpl", None) or globals().get("ver_prompt_anchor", None) or globals().get("verify_prompt_anchor", None)

        # use_anchor = bool(getattr(self, "use_anchor_verifier", False) or globals().get("USE_ANCHOR_VERIFIER", False))
        # if use_anchor or (normal_tpl is None and anchor_tpl is not None):
        #     ver_tpl = anchor_tpl
        # else:
        #     ver_tpl = normal_tpl
        # ver_tpl = ver_tpl or normal_tpl or anchor_tpl
        use_anchor = (VER_PROMPT_MODE == "anchor")
        ver_tpl = verify_prompt_anchor_tpl if use_anchor else verify_prompt_tpl

        user_prompt = _render(
            ver_tpl,
            symptom_text=symptom_text,
            symptom_list=symptom_list_json,
            initial_result=wrapped_result_str,
            candidate_json=candidate_blocks_str,
            # common alternates (anchor templates in your repo use these names)
            NOTE_TEXT=symptom_text,
            SYMPTOM_LIST=symptom_list_json,
            # INITIAL_RESULT=wrapped_result_str,
            CANDIDATE_JSON=candidate_blocks_str,
        )

        return [
            {"role": "system", "content": "You are a medical temporal sequence verification assistant. Return ONLY JSON: {'score':int,'feedback':str}. feedback required if score < 3."},
            {"role": "user", "content": user_prompt},
        ]
    def process_dataframe(self, df_chunk: pd.DataFrame) -> List[Dict[str, Any]]:
        """
        Returns a list of dicts (same order as df_chunk rows) suitable for writing into df_chunk['new_llm_result'].
        """
        _load_vllm_once()

        n = len(df_chunk)
        texts = df_chunk["symptom_text"].tolist()
        sym_lists = [normalize_symptom_list(x) for x in df_chunk["symptom_list"].tolist()]

        # Gate
        gate = [{"has_progression": True, "signals": ["gate_off"], "confidence": 1.0}] * n
        if self.use_temporal_gate:
            gate = temporal_id_batch(texts, min_confidence=self.temporal_conf_threshold)

        # State
        done = [False] * n
        feedback: List[Optional[str]] = [None] * n
        prev_result: List[Optional[Any]] = [None] * n
        final_fixed: List[Optional[Dict[str, Any]]] = [None] * n
        final_score = [0] * n
        status = ["running"] * n
        attempts: List[List[Dict[str, Any]]] = [[] for _ in range(n)]

        # Immediately finalize "no temporal pattern" rows, but keep processing others
        for i in range(n):
            if not gate[i].get("has_progression", True):
                done[i] = True
                final_fixed[i] = None
                final_score[i] = 0
                status[i] = "no_temporal_pattern"
                attempts[i].append({"temporal_id": gate[i]})

        # Iterative extraction for active rows
        for it in range(self.max_iterations):
            active = [i for i in range(n) if not done[i]]
            if not active:
                break

            # ---- GENERATION (batched)
            gen_msgs_batch = []
            for i in active:
                gen_msgs_batch.append(self._build_gen_messages(
                    texts[i], sym_lists[i], 
                    prior_feedback=feedback[i],
                    prior_result=prev_result[i],
                ))

            gen_outs = call_llm_batch(
                gen_msgs_batch,
                temperature=self.gen_temperature,
                max_output_tokens=self.gen_max_tokens,
                micro_batch_size=self.micro_batch_size,
            )

            # parse + format fix (deterministic)
            fixed_lists: List[List[Dict[str, Any]]] = []
            for k, i in enumerate(active):
                raw_gen = gen_outs[k]
                parsed_gen = parse_json_best_effort(raw_gen)

                post = format_check_and_fix(parsed_gen if parsed_gen is not None else raw_gen, sym_lists[i])
                fixed_list = post["fixed"]

                attempts[i].append({
                    "raw_gen": raw_gen,
                    "parsed_gen": parsed_gen,
                    "post_fix": fixed_list,
                    "fix_report": post["report"],
                })

                fixed_lists.append(fixed_list)

            # ---- VERIFICATION (batched, on FIXED JSON)
            ver_msgs_batch = []
            for k, i in enumerate(active):
                ver_msgs_batch.append(self._build_ver_messages(texts[i], sym_lists[i], fixed_lists[k]))

            ver_outs = call_llm_batch(
                ver_msgs_batch,
                temperature=self.ver_temperature,
                max_output_tokens=self.ver_max_tokens,
                micro_batch_size=self.micro_batch_size,
            )

            # update states
            for k, i in enumerate(active):
                raw_ver = ver_outs[k]
                parsed_ver = parse_json_best_effort(raw_ver)

                attempts[i][-1]["raw_verify"] = raw_ver
                attempts[i][-1]["parsed_verify"] = parsed_ver

                score = parsed_ver.get("score") if isinstance(parsed_ver, dict) else None
                if isinstance(score, int):
                    final_score[i] = score
                else:
                    final_score[i] = 0

                if isinstance(score, int) and score >= self.accept_score:
                    done[i] = True
                    final_fixed[i] = {"temporal_result": fixed_lists[k]}
                    status[i] = "ok"
                else:
                    feedback[i] = parsed_ver.get("feedback") if isinstance(parsed_ver, dict) else None
                    prev_result[i] = {"temporal_result": fixed_lists[k]}

        # Finalize unfinished rows
        for i in range(n):
            if status[i] == "running":
                # last fixed result if any; else all-none
                if prev_result[i] is not None:
                    final_fixed[i] = prev_result[i]
                else:
                    final_fixed[i] = {"temporal_result": [{s: "none" for s in sym_lists[i]}]}
                status[i] = "max_iterations_reached"

        # Pack outputs in the same shape you already used downstream
        out: List[Dict[str, Any]] = []
        for i in range(n):
            out.append({
                "temporal_id": gate[i],
                "attempts": attempts[i],
                "final_result": final_fixed[i],
                "score": final_score[i] if status[i] in ("ok", "max_iterations_reached") else None,
                "status": status[i],
            })
        return out





In [19]:
# ----------------------------
# AGENT CALL (writes into pred_source['new_llm_result'])
# ----------------------------
agent = TemporalExtractionBatchAgent()

# Chunk the dataframe to keep memory stable; vLLM batching happens inside each chunk.
CHUNK_ROWS = 500  # adjust; 64~256 is typical. Does NOT change LLM micro-batching.
all_results = [None] * len(sample)

for start in range(0, len(sample), CHUNK_ROWS):
    end = min(start + CHUNK_ROWS, len(sample))
    print(f"Processing rows {start} to {end}...")
    batch_df = sample.iloc[start:end].copy()
    batch_results = agent.process_dataframe(batch_df)

    # store into the original df (as python dicts; writing to CSV will stringify them)
    sample.loc[batch_df.index, "new_llm_result"] = batch_results

    print(f"[done] rows {start}:{end}  ({end}/{len(sample)})")
    sample.to_csv(f"llama70b_premium_updated_with_temporal_part_{start}.csv", index=False)
# pred_source now contains the column "new_llm_result" with attempts + final_result + score/status.
# Your existing evaluation code can remain unchanged.


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


Processing rows 0 to 500...
[vLLM] Loading shuyuej/Llama-3.3-70B-Instruct-GPTQ | tp=2 | dtype=float16 | max_model_len=8192


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


Parse safetensors files:   0%|          | 0/8 [00:00<?, ?it/s]

[2026-03-07 19:55:33] INFO parallel_state.py:1212: world_size=2 rank=0 local_rank=0 distributed_init_method=tcp://127.0.0.1:48469 backend=nccl
[2026-03-07 19:55:33] INFO parallel_state.py:1212: world_size=2 rank=1 local_rank=1 distributed_init_method=tcp://127.0.0.1:48469 backend=nccl
[2026-03-07 19:55:33] INFO pynccl.py:111: vLLM is using nccl==2.27.5
[2026-03-07 19:55:33] WARNING symm_mem.py:67: SymmMemCommunicator: Device capability 8.6 not supported, communicator is not available.
[2026-03-07 19:55:33] WARNING symm_mem.py:67: SymmMemCommunicator: Device capability 8.6 not supported, communicator is not available.
[2026-03-07 19:55:33] INFO parallel_state.py:1423: rank 1 in world size 2 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 1, EP rank N/A
[2026-03-07 19:55:33] INFO parallel_state.py:1423: rank 0 in world size 2 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A
[2026-03-07 19:55:34] INFO gpu_model_runner.py:4033: Starting to load model shuyuej

[done] rows 0:500  (500/3156)
Processing rows 500 to 1000...
[done] rows 500:1000  (1000/3156)
Processing rows 1000 to 1500...
[done] rows 1000:1500  (1500/3156)
Processing rows 1500 to 2000...
[done] rows 1500:2000  (2000/3156)
Processing rows 2000 to 2500...
[done] rows 2000:2500  (2500/3156)
Processing rows 2500 to 3000...
[done] rows 2500:3000  (3000/3156)
Processing rows 3000 to 3156...
[done] rows 3000:3156  (3156/3156)


In [20]:
len(sample)

3156

In [21]:
sample.head()


,vaers_id,symptom_text,symptom_list,llm_result,llm_result_formatted,final_corrected_version,final_result,no_temporal_info,new_llm_result
0,1420769,"About 4-5 days after my second dose, I began h...","['Chest discomfort', 'Chest pain', 'Electrocar...","[ \n { \n ""Chest pain"": [""non stop chest...","[\n {\n ""Chest pain"": [\n ""non stop c...",NaN,1,False,"{'temporal_id': {'has_progression': True, 'sig..."
2,1831772,"Muscle pain, headache, nausea. Muscle pain ins...","['Headache', 'Immediate post-injection reactio...","[\n {\n ""Headache"": [""none""],\n ""Immedi...","[\n {\n ""Myalgia"": [\n ""Muscle pain""\...","[\n {\n ""Myalgia"": [\n ""Muscle pain""\...",0,False,"{'temporal_id': {'has_progression': True, 'sig..."
3,1021935,back pain/soreness in his lower back/had bad l...,"['Arthralgia', 'Back pain', 'Nausea', 'Pain in...","[\n {\n ""Vaccination site pain"": [""injecti...","[\n {\n ""Vaccination site pain"": [\n ...",NaN,1,False,"{'temporal_id': {'has_progression': True, 'sig..."
4,2749020,The above vaccines administered without incide...,"['Dizziness', 'Fall', 'Fatigue', 'Head injury'...","[\n {\n ""Syncope"": [""patient fainted""],\n ...","[\n {\n ""Syncope"": [\n ""patient faint...",NaN,1,False,"{'temporal_id': {'has_progression': True, 'sig..."
5,940509,"about 4 hrs after vaccination ee had ""tunnel ...","['Dizziness', 'Faeces soft', 'Fatigue', 'Flatu...","[\n {\n ""Tunnel vision"": [""tunnel vision""]...","[\n {\n ""Tunnel vision"": [\n ""tunnel ...",NaN,1,False,"{'temporal_id': {'has_progression': True, 'sig..."


In [22]:
# counter = 5
res_lst = []
for row in sample.iterrows():
    final_result = row[1]['new_llm_result']['attempts'][-1]['post_fix']
    # if counter > 0:
    #     print(final_result)
    #     counter -= 1
    # else:
    #     break
    res = json.dumps(final_result, ensure_ascii=False, indent=2)
    res_lst.append(res)

sample['final_result_cleaned'] = res_lst
sample.head()
sample.to_csv('llama70b_premium_updated_with_temporal_final.csv', index=False)

In [23]:
# res_lst = []
# for row in results_df.iterrows():
#     clean_out = parse_json_best_effort(row[1]['attempts_json'])
#     # print(len(clean_out))
#     final_result = clean_out[-1]['post_fix'] if clean_out else None
#     res = json.dumps(final_result, ensure_ascii=False, indent=2)
#     res_lst.append(res)

# # store res_lst in reslts_df
# results_df['final_result_cleaned'] = res_lst
# results_df.head()



In [24]:
# res_lst_3 = []
# for row in results_df.iterrows():
#     len_row = len(row[1]['new_llm_result']['attempts'])
#     if len_row >1:
#         final_result = row[1]['new_llm_result']['attempts'][1]['post_fix']
#     else:
#         final_result = row[1]['new_llm_result']['attempts'][0]['post_fix']
#     # final_result = row[1]['new_llm_result']['attempts'][-1]['post_fix']
 
#     res = json.dumps(final_result, ensure_ascii=False, indent=2)
#     res_lst_3.append(res)

# results_df['final_result_cleaned'] = res_lst_3
# results_df.head()
# results_df.to_csv('openai_our_method_results_with_temporal_final_2_iter.csv', index=False)

In [25]:
# check results_df for final_result_cleaned column and compared to original llm_result_formatted by iterating through the rows and printing both for comparison
# setupt a counter to limit the number of rows printed for comparison
counter = 0
# max_rows = 5  # Limit to 5 rows for comparison

for row in sample.sample(n=10).iterrows():
    # if counter >= max_rows:
    #     break
    print("Original LLM Result Formatted:")
    print(row[1]['llm_result_formatted'])
    print("Final Result Cleaned:")
    print(row[1]['final_result_cleaned'])
    counter += 1


Original LLM Result Formatted:
[
  {
    "Tremor": [
      "intense shaking"
    ],
    "Chills": [
      "chills"
    ],
    "Pyrexia": [
      "fever"
    ],
    "Headache": [
      "headache"
    ]
  },
  {
    "Rash": [
      "Covid arm rash appeared less than 48 hours after injection"
    ]
  }
]
Final Result Cleaned:
[
  {
    "Rash": [
      "Covid arm rash"
    ]
  },
  {
    "Chills": [
      "intense shaking, chills"
    ],
    "Headache": [
      "headache"
    ],
    "Tremor": [
      "intense shaking"
    ]
  },
  {
    "Pyrexia": [
      "fever"
    ]
  }
]
Original LLM Result Formatted:
[
  {
    "Syncope": [
      "Fainted a few mins after receiving the dose"
    ]
  },
  {
    "Fall": [
      "fell to the floor"
    ]
  },
  {
    "Lip injury": [
      "Cut his upper lip during the fall"
    ]
  },
  {
    "Hyperhidrosis": [
      "sweaty forehead and palms"
    ]
  }
]
Final Result Cleaned:
[
  {
    "Syncope": [
      "Fainted a few mins after receiving the dose"
   

# Evaluation

## Exact Match

In [26]:
from typing import Iterable, Callable, Set, Hashable, Tuple, Any
# # exact match
# def strict_exact_match(
#     gold: Iterable[Iterable[Any]],
#     pred: Iterable[Iterable[Any]],
#     normalize: Optional[Callable[[Any], Any]] = None,
# ) -> Tuple[bool, str]:
#     """
#     Strict Exact Match for bucketed sequences:
#       - Compares two sequences of groups/buckets (e.g., [[A,B],[C],[D,E]]).
#       - Bucket order MUST match exactly.
#       - Within-bucket order is ignored (treated as sets).
#       - Duplicate items within a bucket are ignored (set semantics).

#     Args:
#         gold: Iterable of buckets (each bucket is an iterable of items).
#         pred: Iterable of buckets (each bucket is an iterable of items).
#         normalize: Optional normalization function applied to each item
#                    before comparison (e.g., str.lower, str.strip).

#     Returns:
#         (matched, reason)
#         matched: True if strictly equal under the bucket-as-set rule; else False
#         reason:  "" on success; otherwise a brief diagnostic message
#     """
#     # Materialize as list of lists for safe reuse
#     gold_buckets: List[List[Any]] = [list(b) for b in gold]
#     pred_buckets: List[List[Any]] = [list(b) for b in pred]

#     # 1) Same number of buckets
#     if len(gold_buckets) != len(pred_buckets):
#         return (False, f"Different # of buckets: gold={len(gold_buckets)} vs pred={len(pred_buckets)}")

#     # Helper: normalize + set-ify a bucket (ignore within-bucket order, dedupe)
#     def bucket_set(bucket: Iterable[Any]) -> Set[Any]:
#         if normalize is None:
#             return set(bucket)
#         return set(normalize(x) for x in bucket)

#     # 2) Compare each bucket as a set, in order
#     for i, (g_b, p_b) in enumerate(zip(gold_buckets, pred_buckets)):
#         g_set = bucket_set(g_b)
#         p_set = bucket_set(p_b)
#         if g_set != p_set:
#             missing = g_set - p_set
#             extra   = p_set - g_set
#             parts = []
#             if missing:
#                 parts.append(f"missing={sorted(missing)}")
#             if extra:
#                 parts.append(f"extra={sorted(extra)}")
#             details = "; ".join(parts) if parts else "bucket mismatch"
#             return (False, f"Bucket {i} differs: {details}")

#     # 3) All buckets matched
#     return (True, "")

from collections import Counter

def strict_exact_match(
    gold: Iterable[Iterable[Any]],
    pred: Iterable[Iterable[Any]],
    normalize: Optional[Callable[[Any], Any]] = None,
) -> Tuple[bool, str]:
    """
    Strict Exact Match for bucketed sequences:
      - Compares two sequences of groups/buckets (e.g., [[A,B],[C],[D,E]]).
      - Bucket order MUST match exactly.
      - Within-bucket order is ignored.
      - Duplicates MUST match (multiset/bag semantics), i.e., counts per item must be equal.

    Args:
        gold: Iterable of buckets (each bucket is an iterable of items).
        pred: Iterable of buckets (each bucket is an iterable of items).
        normalize: Optional normalization function applied to each item
                   before comparison (e.g., str.lower, str.strip).

    Returns:
        (matched, reason)
        matched: True if strictly equal under bucket-as-multiset; else False
        reason:  "" on success; otherwise a brief diagnostic message
    """
    # Materialize as list of lists for safe reuse
    gold_buckets = [list(b) for b in gold]
    pred_buckets = [list(b) for b in pred]

    # 1) Same number of buckets
    if len(gold_buckets) != len(pred_buckets):
        return (False, f"Different # of buckets: gold={len(gold_buckets)} vs pred={len(pred_buckets)}")

    # Normalizer
    norm = (lambda x: normalize(x)) if normalize is not None else (lambda x: x)

    # 2) Compare each bucket as a multiset (counts must match), in order
    for i, (g_b, p_b) in enumerate(zip(gold_buckets, pred_buckets)):
        g_ctr = Counter(norm(x) for x in g_b)
        p_ctr = Counter(norm(x) for x in p_b)
        if g_ctr != p_ctr:
            # Report missing/extra counts explicitly
            missing = g_ctr - p_ctr   # items required but missing counts
            extra   = p_ctr - g_ctr   # items present in pred but not in gold (or too many)
            parts = []
            if missing:
                parts.append(f"missing={dict(sorted(missing.items()))}")
            if extra:
                parts.append(f"extra={dict(sorted(extra.items()))}")
            details = "; ".join(parts) if parts else "bucket mismatch"
            return (False, f"Bucket {i} differs: {details}")

    # 3) All buckets matched
    return (True, "")

In [27]:
# ---- Example ----
gold = [["fever", "headache"], ["rash"], ["fatigue", "nausea"]]
pred = [["rash"],["headache", "fever"], ["nausea", "fatigue"]]
strict_exact_match(gold, pred, normalize=lambda s: s.strip().lower())

(False,
 "Bucket 0 differs: missing={'fever': 1, 'headache': 1}; extra={'rash': 1}")

In [28]:
# ---- Example 1: Normalized match succeeds ----
gold = [["Dose 1", "Dose 2"], ["Observation", "Observation"]]
pred = [["dose 2", " dose 1 "], [" observation ", "Observation"]]
strict_exact_match(gold, pred, normalize=lambda s: s.strip().lower())

(True, '')

In [29]:
# ---- Example 2: Bucket count mismatch ----
gold = [["fever"], ["rash"]]
pred = [["fever"], ["rash"], ["cough"]]
strict_exact_match(gold, pred, normalize=None)

(False, 'Different # of buckets: gold=2 vs pred=3')

In [30]:
# ---- Example 3: Duplicate counts must match ----
gold = [["fever", "fever"], ["rash"]]
pred = [["fever"], ["rash"]]
strict_exact_match(gold, pred, normalize=None)

(False, "Bucket 0 differs: missing={'fever': 1}")

In [31]:
# ---- Example 4: Bucket contents differ ----
gold = [["fever"], ["rash", "cough"]]
pred = [["fever"], ["rash", "hives"]]
strict_exact_match(gold, pred, normalize=None)

(False, "Bucket 1 differs: missing={'cough': 1}; extra={'hives': 1}")

In [32]:
# ------------------ Test Cases ------------------
tests = [
    {
        "name": "1) Perfect match; within-bucket order ignored",
        "gold": [['fever','headache'], ['rash'], ['fatigue','nausea']],
        "pred": [['headache','fever'], ['rash'], ['nausea','fatigue']],
    },
    {
        "name": "2) Different number of buckets",
        "gold": [['fever','headache'], ['rash']],
        "pred": [['fever','headache'], ['rash'], ['fatigue']],
    },
    {
        "name": "3) Extra item in pred (bucket 0)",
        "gold": [['fever','headache'], ['rash']],
        "pred": [['fever','headache','cough'], ['rash']],
    },
    {
        "name": "4) Missing item in pred (bucket 0)",
        "gold": [['fever','headache'], ['rash']],
        "pred": [['fever'], ['rash']],
    },
    {
        "name": "5) Buckets swapped (order mismatch)",
        "gold": [['fever'], ['rash']],
        "pred": [['rash'], ['fever']],
    },
    {
        "name": "6) Duplicates within bucket are ignored",
        "gold": [['fever','fever','headache'], ['rash']],
        "pred": [['headache','fever'], ['rash']],
    },
    {
        "name": "7) Pred has an extra empty bucket",
        "gold": [['fever','headache'], ['rash']],
        "pred": [['fever','headache'], ['rash'], []],
    },
    {
        "name": "8) Segmentation error across buckets",
        "gold": [['fever','headache'], ['rash']],
        "pred": [['fever'], ['headache','rash']],
    },
    {
        "name": "9) Case/whitespace differences handled by normalize",
        "gold": [['Fever',' Headache '], ['RASH']],
        "pred": [['fever','headache'], ['rash']],
    },
]

# ------------------ Run Tests ------------------
for t in tests:
    match, reason = strict_exact_match(t["gold"], t["pred"], normalize=lambda s: s.strip().lower())
    print(f"{t['name']}\n  gold={t['gold']}\n  pred={t['pred']}\n  -> match={match}, reason={reason}\n")


1) Perfect match; within-bucket order ignored
  gold=[['fever', 'headache'], ['rash'], ['fatigue', 'nausea']]
  pred=[['headache', 'fever'], ['rash'], ['nausea', 'fatigue']]
  -> match=True, reason=

2) Different number of buckets
  gold=[['fever', 'headache'], ['rash']]
  pred=[['fever', 'headache'], ['rash'], ['fatigue']]
  -> match=False, reason=Different # of buckets: gold=2 vs pred=3

3) Extra item in pred (bucket 0)
  gold=[['fever', 'headache'], ['rash']]
  pred=[['fever', 'headache', 'cough'], ['rash']]
  -> match=False, reason=Bucket 0 differs: extra={'cough': 1}

4) Missing item in pred (bucket 0)
  gold=[['fever', 'headache'], ['rash']]
  pred=[['fever'], ['rash']]
  -> match=False, reason=Bucket 0 differs: missing={'headache': 1}

5) Buckets swapped (order mismatch)
  gold=[['fever'], ['rash']]
  pred=[['rash'], ['fever']]
  -> match=False, reason=Bucket 0 differs: missing={'fever': 1}; extra={'rash': 1}

6) Duplicates within bucket are ignored
  gold=[['fever', 'fever', 'h

[True, False, False, False, False, True, False, False, True]

## Group-aware Longest Common Contiguous Subsequence

In [33]:

def group_aware_lccs(
    gold: Iterable[Iterable[Any]],
    pred: Iterable[Iterable[Any]],
    normalize: Optional[Callable[[Any], Any]] = None,
    return_alignment: bool = True,
) -> Tuple[int, float, float, float, Optional[Tuple[Tuple[int,int], Tuple[int,int]]]]:
    """
    Group-aware Longest Common Contiguous Subsequence (LCCS).

    Buckets are treated as sets (order-insensitive inside a bucket).
    We compute the longest block of consecutive buckets that appears in both sequences.

    Returns:
        (lccs_len, recall_gold, precision_pred, f1, alignment)
          - lccs_len: number of buckets in the longest contiguous match
          - recall_gold: lccs_len / max(1, len(gold_tokens))
          - precision_pred: lccs_len / max(1, len(pred_tokens))
          - f1: F1 over bucket-block matching using the above precision/recall
          - alignment: ((g_start, g_end_inclusive), (p_start, p_end_inclusive)) or None
    """
    def canon_bucket(bucket: Iterable[Any]) -> Hashable:
        if normalize is None:
            return frozenset(bucket)
        return frozenset(normalize(x) for x in bucket)

    gold_tokens: List[Hashable] = [canon_bucket(b) for b in gold]
    pred_tokens: List[Hashable] = [canon_bucket(b) for b in pred]
    m, n = len(gold_tokens), len(pred_tokens)

    # dp[i][j] = length of longest common suffix ending at gold[i-1], pred[j-1]
    dp = [[0]*(n+1) for _ in range(m+1)]
    best_len = 0
    best_end_g = 0
    best_end_p = 0

    for i in range(1, m+1):
        gi = gold_tokens[i-1]
        for j in range(1, n+1):
            if gi == pred_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
                if dp[i][j] > best_len:
                    best_len = dp[i][j]
                    best_end_g = i
                    best_end_p = j
            else:
                dp[i][j] = 0

    lccs_len = best_len
    recall_gold = lccs_len / max(1, m)
    precision_pred = lccs_len / max(1, n)
    f1 = (2 * recall_gold * precision_pred / (recall_gold + precision_pred)) if (recall_gold + precision_pred) else 0.0

    if not return_alignment or lccs_len == 0:
        return lccs_len, recall_gold, precision_pred, f1, None

    g_start = best_end_g - lccs_len
    g_end   = best_end_g - 1
    p_start = best_end_p - lccs_len
    p_end   = best_end_p - 1
    return lccs_len, recall_gold, precision_pred, f1, ((g_start, g_end), (p_start, p_end))




In [34]:
# ---- Example ----
gold = [["fever","headache"], ["rash"], ["fatigue","nausea"]]
pred = [["headache","fever"], ["cough"], ["rash"], ["nausea","fatigue"]]
l, rg, pp, f1, span = group_aware_lccs(gold, pred, normalize=lambda s: s.strip().lower(), return_alignment=True)
print(l, rg, pp, f1, span)  # -> 2, 2/3, 2/4, ~0.57, ((1,2),(2,3))


2 0.6666666666666666 0.5 0.5714285714285715 ((1, 2), (2, 3))


In [35]:
tests = [
    # 1
    {
        "name": "1) Perfect contiguous match",
        "gold": [['A','B'], ['C'], ['D','E']],
        "pred": [['B','A'], ['C'], ['E','D']],
        "expected_len": 3
    },
    # 2
    {
        "name": "2) Middle bucket mismatch",
        "gold": [['A','B'], ['C'], ['D','E']],
        "pred": [['B','A'], ['X'], ['E','D']],
        "expected_len": 1
    },
    # 3
    {
        "name": "3) Extra bucket inserted but 2-run remains",
        "gold": [['A','B'], ['C'], ['D','E']],
        "pred": [['B','A'], ['C'], ['Y'], ['E','D']],
        "expected_len": 2
    },
    # 4
    {
        "name": "4) Buckets reordered",
        "gold": [['A','B'], ['C'], ['D','E']],
        "pred": [['C'], ['B','A'], ['D','E']],
        "expected_len": 1
    },
    # 5
    {
        "name": "5) No common buckets",
        "gold": [['A','B'], ['C'], ['D','E']],
        "pred": [['X'], ['Y'], ['Z']],
        "expected_len": 0
    },
    # 6
    {
        "name": "6) Two-bucket run at the end",
        "gold": [['A','B'], ['C'], ['D','E']],
        "pred": [['X'], ['B','A'], ['C'], ['Z']],
        "expected_len": 2
    },
    # 7
    {
        "name": "7) Perfect run surrounded by extras",
        "gold": [['A','B'], ['C'], ['D','E']],
        "pred": [['Y'], ['B','A'], ['C'], ['D','E'], ['Z']],
        "expected_len": 3
    },
    # 8
    {
        "name": "8) Split a gold bucket into two predicted buckets",
        "gold": [['A','B'], ['C'], ['D','E']],
        "pred": [['B','A'], ['C'], ['D'], ['E']],
        "expected_len": 2
    },
    # 9
    {
        "name": "9) Merge adjacent gold buckets in prediction",
        "gold": [['A','B'], ['C'], ['D','E']],
        "pred": [['A','B','C'], ['D','E']],
        "expected_len": 1
    },
    # 10
    {
        "name": "10) Case/whitespace differences handled by normalize",
        "gold": [['A','B'], ['C'], ['D','E']],
        "pred": [[' b ',' a '], [' c '], [' d ',' e ']],
        "expected_len": 3
    },
]

# ------------------ Run Tests ------------------
computed_lengths = []
for t in tests:
    l, rg, pp, f1, span = group_aware_lccs(
        t["gold"], t["pred"], normalize=lambda s: str(s).strip().lower(), return_alignment=True
    )
    computed_lengths.append(l)
    status = "PASS" if l == t["expected_len"] else "FAIL"
    print(f"{t['name']} -> LCCS length = {l} (expected {t['expected_len']}) [{status}]")

print("\nComputed LCCS lengths list:")
print(computed_lengths)

1) Perfect contiguous match -> LCCS length = 3 (expected 3) [PASS]
2) Middle bucket mismatch -> LCCS length = 1 (expected 1) [PASS]
3) Extra bucket inserted but 2-run remains -> LCCS length = 2 (expected 2) [PASS]
4) Buckets reordered -> LCCS length = 1 (expected 1) [PASS]
5) No common buckets -> LCCS length = 0 (expected 0) [PASS]
6) Two-bucket run at the end -> LCCS length = 2 (expected 2) [PASS]
7) Perfect run surrounded by extras -> LCCS length = 3 (expected 3) [PASS]
8) Split a gold bucket into two predicted buckets -> LCCS length = 2 (expected 2) [PASS]
9) Merge adjacent gold buckets in prediction -> LCCS length = 1 (expected 1) [PASS]
10) Case/whitespace differences handled by normalize -> LCCS length = 3 (expected 3) [PASS]

Computed LCCS lengths list:
[3, 1, 2, 1, 0, 2, 3, 2, 1, 3]


## Position Error Distance-aware

In [36]:
from collections import defaultdict

def position_error_distance(
    gold: Iterable[Iterable[Any]],
    pred: Iterable[Iterable[Any]],
    normalize: Optional[Callable[[Any], Any]] = None,
    missing_penalty: str = "max",   # "ignore" | "max"
    return_details: bool = False,
) -> Tuple[float, float, Dict[str, Any]]:
    """
    Distance-aware Position Error for bucketed sequences (tie-aware).
    - Assign each item a bucket rank = its bucket index (0-based).
    - Compute absolute bucket-distance for items (ties inside a bucket share rank).
    - Normalize by the maximum possible distance (max(#buckets)-1) to map to [0,1].
    - Optionally penalize missing/extra items with worst-case distance.

    Args:
        gold: sequence of gold buckets, e.g., [["fever","headache"], ["rash"], ["fatigue","nausea"]]
        pred: sequence of predicted buckets
        normalize: optional function to normalize items before comparison (e.g., str.lower)
        missing_penalty: 
            - "ignore": compute over intersection only (no penalty for missing/extra)
            - "max": include union; items missing in one side get worst-case distance
        return_details: if True, returns a details dict with per-item distances and counts

    Returns:
        (score, mae_norm, details)
            score: 1 - mae_norm, clipped to [0,1]  (higher is better)
            mae_norm: normalized mean absolute error in [0,1]
            details: dict with diagnostics if return_details=True, else {}

    Notes:
        - Within-bucket order is ignored (tie-aware).
        - If an item appears in multiple buckets (shouldn't), earliest bucket index is used.
        - If both sequences have only 1 bucket, maximum possible distance is treated as 1 to avoid /0.
    """
    # Helper to normalize an item
    def norm(x: Any) -> Any:
        return x if normalize is None else normalize(x)

    # Map item -> earliest bucket index
    def item_to_rank(seq: Iterable[Iterable[Any]]) -> Dict[Any, int]:
        rank_map: Dict[Any, int] = {}
        for b_idx, bucket in enumerate(seq):
            for x in bucket:
                nx = norm(x)
                # keep earliest occurrence if duplicates across buckets
                if nx not in rank_map:
                    rank_map[nx] = b_idx
        return rank_map

    gold_ranks = item_to_rank(gold)
    pred_ranks = item_to_rank(pred)

    # Determine maximum possible distance between any two bucket ranks
    # B = max(len(list(gold)), len(list(pred)))  # we’ll reuse original iterables shortly; cast once more safely below
    
    # Re-materialize to avoid exhausting iterables
    gold_buckets: List[List[Any]] = [list(b) for b in gold]
    pred_buckets: List[List[Any]] = [list(b) for b in pred]
    B = max(len(gold_buckets), len(pred_buckets))
    max_dist = max(1, B - 1)  # avoid division by zero; if B==1, treat max_dist as 1

    gold_items = set(gold_ranks.keys())
    pred_items = set(pred_ranks.keys())
    inter = gold_items & pred_items
    only_gold = gold_items - pred_items
    only_pred = pred_items - gold_items

    distances: Dict[Any, int] = {}

    # Intersection: real distances
    for x in inter:
        d = abs(pred_ranks[x] - gold_ranks[x])
        distances[x] = d

    # Missing / extra handling
    if missing_penalty == "max":
        # assign worst-case distance to items missing on one side
        for x in only_gold:
            distances[x] = max_dist
        for x in only_pred:
            distances[x] = max_dist
    elif missing_penalty == "ignore":
        pass
    else:
        raise ValueError('missing_penalty must be "ignore" or "max"')

    if len(distances) == 0:
        has_any_gold = any(len(b) > 0 for b in gold_buckets)
        has_any_pred = any(len(b) > 0 for b in pred_buckets)

        # Case A: both sequences empty -> trivially perfect (no displacement possible)
        if not has_any_gold and not has_any_pred:
            score, mae_norm = 1.0, 0.0
            note = "both gold and pred empty; no displacement possible"
        else:
            # Case B: at least one side non-empty but no intersection -> worst score
            score, mae_norm = 0.0, 1.0
            note = "no shared items; intersection-based position treated as worst"

        details = {} if not return_details else {
            "num_gold_buckets": len(gold_buckets),
            "num_pred_buckets": len(pred_buckets),
            "counts": {
                "intersection": 0, 
                "only_gold": len(only_gold), 
                "only_pred": len(only_pred)
                },
            "max_dist": max_dist,
            "mae_raw": score,
            "per_item_distance": {},
            "note": note
        }
        return score, mae_norm, details

    mae = sum(distances.values()) / len(distances)
    mae_norm = min(1.0, mae / max_dist)
    score = max(0.0, 1.0 - mae_norm)

    details = {}
    if return_details:
        details = {
            "num_gold_buckets": len(gold_buckets),
            "num_pred_buckets": len(pred_buckets),
            "counts": {
                "intersection": len(inter),
                "only_gold": len(only_gold),
                "only_pred": len(only_pred),
                "evaluated_items": len(distances)
            },
            "max_dist": max_dist,
            "mae_raw": mae,
            "per_item_distance": dict(sorted(distances.items(), key=lambda kv: (-kv[1], str(kv[0])))),
            "note": None
        }

    return score, mae_norm, details


In [37]:

# ----------------- Example -----------------
gold = [["fever","headache"], ["rash"], ["fatigue","nausea"]]
pred = [["headache","fever"], ["cough"], ["nausea","fatigue"]]
score, mae_norm, info = position_error_distance(
    gold, pred, normalize=lambda s: s.strip().lower(), missing_penalty="max", return_details=True
)
print(score, mae_norm)
print(info)

0.6666666666666667 0.3333333333333333
{'num_gold_buckets': 3, 'num_pred_buckets': 3, 'counts': {'intersection': 4, 'only_gold': 1, 'only_pred': 1, 'evaluated_items': 6}, 'max_dist': 2, 'mae_raw': 0.6666666666666666, 'per_item_distance': {'cough': 2, 'rash': 2, 'fatigue': 0, 'fever': 0, 'headache': 0, 'nausea': 0}, 'note': None}


In [38]:
def run_case(name, gold, pred):
    print(f"\n{name}")
    print("gold:", gold)
    print("pred:", pred)

    s_int, mae_int, d_int = position_error_distance(
        gold, pred, normalize=lambda s: str(s).strip().lower(), missing_penalty="ignore", return_details=True
    )
    print(f"  [intersection] score={s_int:.3f}, mae_norm={mae_int:.3f}, max_dist={d_int['max_dist']} | counts={d_int['counts']} | note = {d_int['note']}")
    print(f"    per-item distances (intersection): {d_int['per_item_distance']}")

    s_uni, mae_uni, d_uni = position_error_distance(
        gold, pred, normalize=lambda s: str(s).strip().lower(), missing_penalty="max", return_details=True
    )
    print(f"  [union]        score={s_uni:.3f}, mae_norm={mae_uni:.3f}, max_dist={d_uni['max_dist']} | counts={d_uni['counts']}| note = {d_int['note']}")
    print(f"    per-item distances (union): {d_uni['per_item_distance']}")


# ------------------ Tests ------------------
gold1 = [["fever","headache"], ["rash"], ["fatigue","nausea"]]
pred1 = [["headache","fever"], ["rash"], ["nausea","fatigue"]]
run_case("Test 1: perfect match", gold1, pred1)

pred2 = [["headache"], ["fever","rash"], ["nausea","fatigue"]]
run_case("Test 2: one item shifted by 1 bucket", gold1, pred2)

pred3 = [["headache"], ["rash"], ["nausea","fatigue","fever"]]
run_case("Test 3: one item shifted by 2 buckets", gold1, pred3)

pred4 = [["headache","fever"], [], ["nausea","fatigue","cough"]]
run_case("Test 4: missing gold item + extra pred item", gold1, pred4)

pred5 = [["headache","fever"], ["rash","cough"], ["nausea","fatigue"]]
run_case("Test 5: extra item only", gold1, pred5)

gold6 = [["a","b"], ["c"], ["d","e"]]
pred6 = [["a","b"], ["d","e"], ["c"]]
run_case("Test 6: swap two buckets", gold6, pred6)

gold7 = [["a","b"], ["c"]]
pred7 = [["a","c"], ["b","c"]]
run_case("Test 7: duplicate in pred (earliest occurrence used)", gold7, pred7)

gold8 = [[" Fever ","Headache "], ["RASH"]]
pred8 = [["fever","headache"], ["rash"]]
run_case("Test 8: normalization check", gold8, pred8)

gold9 = [["a","b","c"]]
pred9 = [["1","2","3","x"]]
run_case("Test 9: single-bucket edge case", gold9, pred9)

gold10 = [["a"], ["b"], ["c"], ["d"]]
pred10 = [["a","b","c","d"]]
run_case("Test 10: different #buckets changes normalization", gold10, pred10)

gold11 = []
pred11 = []
run_case("Test 11: empty for both", gold11, pred11)

# 12: non-empty but no intersection
gold12 = [["a"], ["b"]]
pred12 = [["x"], ["y"]]
run_case("Test 12: non-empty but no intersection", gold12, pred12)

# 13: one empty, one non-empty
gold13 = []
pred13 = [["a"], ["b"]]
run_case("Test 13: one empty, one non-empty", gold13, pred13)




Test 1: perfect match
gold: [['fever', 'headache'], ['rash'], ['fatigue', 'nausea']]
pred: [['headache', 'fever'], ['rash'], ['nausea', 'fatigue']]
  [intersection] score=1.000, mae_norm=0.000, max_dist=2 | counts={'intersection': 5, 'only_gold': 0, 'only_pred': 0, 'evaluated_items': 5} | note = None
    per-item distances (intersection): {'fatigue': 0, 'fever': 0, 'headache': 0, 'nausea': 0, 'rash': 0}
  [union]        score=1.000, mae_norm=0.000, max_dist=2 | counts={'intersection': 5, 'only_gold': 0, 'only_pred': 0, 'evaluated_items': 5}| note = None
    per-item distances (union): {'fatigue': 0, 'fever': 0, 'headache': 0, 'nausea': 0, 'rash': 0}

Test 2: one item shifted by 1 bucket
gold: [['fever', 'headache'], ['rash'], ['fatigue', 'nausea']]
pred: [['headache'], ['fever', 'rash'], ['nausea', 'fatigue']]
  [intersection] score=0.900, mae_norm=0.100, max_dist=2 | counts={'intersection': 5, 'only_gold': 0, 'only_pred': 0, 'evaluated_items': 5} | note = None
    per-item distances 

## Kendall-Tau-b 

In [39]:
import math
from itertools import combinations

def _norm_fn(normalize: Optional[Callable[[Any], Any]]):
    return (lambda x: x) if normalize is None else normalize

def _item_set(seq: Iterable[Iterable[Any]], normalize: Optional[Callable[[Any], Any]]) -> set:
    """All unique items across buckets (order ignored)."""
    norm = _norm_fn(normalize)
    return {norm(x) for bucket in seq for x in bucket}

def _item_ranks(seq: Iterable[Iterable[Any]], normalize: Optional[Callable[[Any], Any]]) -> Dict[Any, int]:
    """
    Map item -> bucket index (rank). Within-bucket order is ignored.
    Assumes no cross-bucket duplicates; if they exist, earliest index wins.
    """
    norm = _norm_fn(normalize)
    ranks: Dict[Any, int] = {}
    for b_idx, bucket in enumerate(seq):
        seen = set()
        for x in bucket:
            k = norm(x)
            if k in seen:   # ignore within-bucket duplicates
                continue
            seen.add(k)
            if k not in ranks:
                ranks[k] = b_idx
    return ranks

# ---------------- Coverage: item set P/R/F1 ----------------

def item_set_prf1(
    gold: Iterable[Iterable[Any]],
    pred: Iterable[Iterable[Any]],
    normalize: Optional[Callable[[Any], Any]] = None,
) -> Tuple[float, float, float, Dict[str, int]]:
    """Coverage only (ignores order)."""
    gold_items = _item_set(gold, normalize)
    pred_items = _item_set(pred, normalize)
    inter = gold_items & pred_items

    precision = len(inter) / len(pred_items) if pred_items else 1.0
    recall    = len(inter) / len(gold_items) if gold_items else 1.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    details = {
        "gold_count": len(gold_items),
        "pred_count": len(pred_items),
        "intersection": len(inter),
        "only_gold": len(gold_items - pred_items),
        "only_pred": len(pred_items - gold_items),
    }
    return precision, recall, f1, details

# ---------------- Order: tie-aware Kendall’s τ-b (intersection) ----------------

def kendall_tau_b_ties(
    gold: Iterable[Iterable[Any]],
    pred: Iterable[Iterable[Any]],
    normalize: Optional[Callable[[Any], Any]] = None,
    return_details: bool = False,
) -> Tuple[float, Dict[str, Any]]:
    """
    Tie-aware Kendall's τ-b on bucketed sequences.
    - Rank = bucket index; items in same bucket are 'tied'.
    - Only items present in BOTH are compared (pure ordering on intersection).
    """
    gold_r = _item_ranks(gold, normalize)
    pred_r = _item_ranks(pred, normalize)

    items = sorted(gold_r.keys() & pred_r.keys(), key=lambda x: str(x))
    n = len(items)
    if n < 2:
        return 1.0, ({"num_items_intersection": n, "pairs": 0} if return_details else {})

    C = D = T_g = T_p = 0  # concordant, discordant, ties in gold, ties in pred
    disagreements: List[Tuple[Any, Any, int, int]] = []

    for i in range(n):
        a = items[i]
        ga = gold_r[a]; pa = pred_r[a]
        for j in range(i+1, n):
            b = items[j]
            gb = gold_r[b]; pb = pred_r[b]

            # relations: -1 (a before b), 0 (tie), +1 (a after b)
            rg = -1 if ga < gb else (1 if ga > gb else 0)
            rp = -1 if pa < pb else (1 if pa > pb else 0)

            if rg == 0: T_g += 1
            if rp == 0: T_p += 1

            if rg != 0 and rp != 0:
                if rg == rp:
                    C += 1
                else:
                    D += 1
                    if len(disagreements) < 20:
                        disagreements.append((a, b, rg, rp))

    numer = C - D
    denom_left  = C + D + T_g
    denom_right = C + D + T_p
    denom = math.sqrt(denom_left * denom_right) if denom_left > 0 and denom_right > 0 else 0.0
    tau_b = (numer / denom) if denom > 0 else 1.0

    if not return_details:
        return tau_b, {}

    return tau_b, {
        "num_items_intersection": n,
        "pairs": n * (n - 1) // 2,
        "C": C, "D": D, "T_g": T_g, "T_p": T_p,
        "numerator": numer,
        "denominator_parts": {"left": denom_left, "right": denom_right},
        "tau_b": tau_b,
        "sample_disagreements": disagreements,
        "relation_legend": {"-1": "a precedes b", "0": "tie", "+1": "a follows b"},
    }

# ---------------- Optional: combine Order × Coverage ----------------

def combined_order_coverage(
    tau_b: float,
    item_f1: float,
    method: str = "product",   # "product" | "geom" | "weighted_geom"
    weight_tau: float = 0.6,   # only used for "weighted_geom"
    weight_f1: float = 0.4,
) -> float:
    """
    Roll up order (τ-b) and coverage (item F1) into one number in [0,1].
    """
    tau = max(0.0, min(1.0, tau_b))
    f1  = max(0.0, min(1.0, item_f1))
    if method == "product":
        return tau * f1
    if method == "geom":
        return math.sqrt(tau * f1)
    if method == "weighted_geom":
        wsum = max(1e-9, weight_tau + weight_f1)
        # (tau^w1 * f1^w2)^(1/(w1+w2))
        return (tau ** weight_tau) * (f1 ** weight_f1) ** (1.0 / wsum)
    raise ValueError('method must be "product", "geom", or "weighted_geom"')

# ---------------- Convenience wrapper ----------------

def evaluate_order_and_coverage(
    gold: Iterable[Iterable[Any]],
    pred: Iterable[Iterable[Any]],
    normalize: Optional[Callable[[Any], Any]] = None,
    combine: Optional[str] = None,  # None | "product" | "geom" | "weighted_geom"
) -> Dict[str, Any]:
    """
    Returns:
      - tau_b (order on intersection, tie-aware)
      - item_precision, item_recall, item_f1 (coverage)
      - combined (optional)
      - details (diagnostics)
    """
    tau, tau_info = kendall_tau_b_ties(gold, pred, normalize=normalize, return_details=True)
    P, R, F1, cov_info = item_set_prf1(gold, pred, normalize=normalize)

    out = {
        "tau_b": tau,
        "item_precision": P,
        "item_recall": R,
        "item_f1": F1,
        "order_details": tau_info,
        "coverage_details": cov_info,
    }
    if combine:
        out["combined"] = combined_order_coverage(tau, F1, method=combine)
        out["combined_method"] = combine
    return out




In [40]:
# ---------------- Example ----------------
gold = [["Fever","Headache"], ["Rash"], ["Fatigue","Nausea"]]
pred = [["headache","fever"], ["Cough"], ["nausea","fatigue"]]
res = evaluate_order_and_coverage(
    gold, pred, normalize=lambda s: s.strip().lower(), combine="product"
)
print(res)

{'tau_b': 0.6666666666666666, 'item_precision': 0.8, 'item_recall': 0.8, 'item_f1': 0.8000000000000002, 'order_details': {'num_items_intersection': 4, 'pairs': 6, 'C': 4, 'D': 0, 'T_g': 2, 'T_p': 2, 'numerator': 4, 'denominator_parts': {'left': 6, 'right': 6}, 'tau_b': 0.6666666666666666, 'sample_disagreements': [], 'relation_legend': {'-1': 'a precedes b', '0': 'tie', '+1': 'a follows b'}}, 'coverage_details': {'gold_count': 5, 'pred_count': 5, 'intersection': 4, 'only_gold': 1, 'only_pred': 1}, 'combined': 0.5333333333333334, 'combined_method': 'product'}


## Sanity Check

In [41]:
# # =========================
# # LCCS, Kendall's tau-b, and PED unit checks (3-bucket cases)
# # Assumes these functions already exist in the notebook scope:
# #   group_aware_lccs, evaluate_order_and_coverage, position_error_distance
# # =========================

# import pandas as pd
# import math

# def get_tau_b(oc: dict) -> float:
#     """Pull tau_b from your evaluate_order_and_coverage return dict."""
#     if isinstance(oc, dict) and "tau_b" in oc and isinstance(oc["tau_b"], (int, float)):
#         return float(oc["tau_b"])
#     od = oc.get("order_details", {}) if isinstance(oc, dict) else {}
#     if isinstance(od, dict) and "tau_b" in od and isinstance(od["tau_b"], (int, float)):
#         return float(od["tau_b"])
#     return float("nan")

# # ---------- Test set ----------
# # Notation:
# #   Gold groups: G1=[...], G2=[...], G3=[...]
# #   Pred  groups: P1=[...], P2=[...], P3=[...]
# # Items: A,B,C,D,E
# # For τ-b calculations:
# #   n = 5 items -> total unordered pairs n0 = 5*4/2 = 10
# #   S = C - D (concordant minus discordant over NON-tied pairs)
# #   n1 = ties in gold (sum over groups of comb(size,2))
# #   n2 = ties in pred
# #   τ_b = S / sqrt((n0 - n1) * (n0 - n2))
# #
# # For LCCS (group-aware but item-based contiguous subsequence intuition):
# #   We flatten to item sequences respecting group order & within-group order as written below,
# #   then take the Longest Common Contiguous Subsequence length = L.
# #   precision = L / len(pred_items), recall = L / len(gold_items), F1 = 2PR/(P+R).
# #
# # For PED (Position Error Distance; illustrative/ideal MAE over group indices):
# #   distance for an item = |group_index_pred - group_index_gold|
# #   MAE = sum(distances)/count(items)
# #   (Your function returns two floats; we show the ideal MAE as a reference.)

# tests = [

#     # 1) Perfect match
#     dict(
#         name="X1_perfect",
#         gold=[["A","B"], ["C"], ["D","E"]],
#         pred=[["A","B"], ["C"], ["D","E"]],
#         note="""
# τ-b: perfect agreement => C=8, D=0, ties in both: n1=n2=2 (AB, DE), n0=10
#      τ_b = (8-0)/sqrt((10-2)*(10-2)) = 8/8 = 1.0
# LCCS: gold=[A,B,C,D,E], pred=[A,B,C,D,E] => L=5, P=1, R=1, F1=1
# PED:  all items stay in the same group => MAE=0
# """
#     ),

#     # 2) Swap the last two groups (adjacent swap)
#     dict(
#         name="X2_swap_G2_G3",
#         gold=[["A","B"], ["C"], ["D","E"]],
#         pred=[["A","B"], ["D","E"], ["C"]],
#         note="""
# τ-b: pairs (A,B) & (D,E) are ties in both => n1=n2=2; n0=10
#      Check signs: only (C vs D) and (C vs E) flip => C=7, D=2 -> S=5
#      τ_b = 5 / sqrt((10-2)*(10-2)) = 5/8 = 0.625
# LCCS: gold seq [A,B,C,D,E], pred seq [A,B,D,E,C] => longest common contiguous block is [A,B] (L=2)
#      P=2/5=0.4, R=2/5=0.4, F1=0.4
# PED:  group indices: A,B:1→1 (0); C:2→3 (1); D,E:3→2 (1 each)
#      MAE = (0+0+1+1+1)/5 = 0.6
# """
#     ),

#     # 3) Merge C with D,E into one group in pred
#     dict(
#         name="X3_merge_tail",
#         gold=[["A","B"], ["C"], ["D","E"]],
#         pred=[["A","B"], ["C","D","E"]],
#         note="""
# τ-b: n0=10; gold ties (AB, DE) => n1=2; pred ties (AB plus C,D,E all tied except AB) => n2=4
#      Non-tied both: A/B vs C/D/E (6 pairs) all concordant; others tied in one ranking
#      S=6 => τ_b = 6 / sqrt((10-2)*(10-4)) = 6 / sqrt(8*6) = 6/√48 ≈ 0.8660254
# LCCS: gold [A,B,C,D,E], pred [A,B,C,D,E] (contiguous order preserved) => L=5, P=R=F1=1
# PED:  D,E moved from group 3 -> 2 (1 each); others unchanged => MAE=(0+0+0+1+1)/5 = 0.4
# """
#     ),

#     # 4) Split first group in pred (A alone), tie C with D,E => bigger tie set in pred
#     dict(
#         name="X4_split_head",
#         gold=[["A","B"], ["C"], ["D","E"]],
#         pred=[["A"], ["B"], ["C","D","E"]],
#         note="""
# τ-b: n0=10; gold ties: (AB, DE) => n1=2; pred ties: (C,D,E) => 3 pairs, n2=3
#      Concordant: A vs C/D/E (3), B vs C/D/E (3) => C=6, D=0 => S=6
#      τ_b = 6 / sqrt((10-2)*(10-3)) = 6 / √56 ≈ 0.8017837
# LCCS: gold [A,B,C,D,E], pred [A,B,C,D,E] => L=5, P=R=F1=1
# PED:  B: 1→2 (1); C: 2→3 (1); A,D,E unchanged (0) => MAE=(0+1+1+0+0)/5 = 0.4
# """
#     ),

#     # 5) Strong reversal of order (all groups reversed)
#     dict(
#         name="X5_full_reverse",
#         gold=[["A","B"], ["C"], ["D","E"]],
#         pred=[["D","E"], ["C"], ["A","B"]],
#         note="""
# τ-b: complete reversal except within-group ties; n0=10; n1=n2=2
#      All 8 cross-group pairs discordant => C=0, D=8 => S=-8
#      τ_b = -8 / sqrt((10-2)*(10-2)) = -8/8 = -1.0
# LCCS: gold [A,B,C,D,E], pred [D,E,C,A,B] => best contiguous match is [C] => L=1
#      P=R=1/5=0.2, F1=0.2
# PED:  A,B: 1→3 (2 each); C: 2→2 (0); D,E: 3→1 (2 each) => MAE=(2+2+0+2+2)/5 = 1.6
# """
#     ),

#     # 6) Move C up into the first group in pred (order preserved in items)
#     dict(
#         name="X6_pull_C_up",
#         gold=[["A","B"], ["C"], ["D","E"]],
#         pred=[["A","B","C"], ["D"], ["E"]],
#         note="""
# τ-b: n0=10; gold ties: (AB, DE) => n1=2; pred ties: (A,B,C) => 3 pairs, n2=3
#      Non-tied both: A/B vs D/E (4) + C vs D/E (2) => C=6, D=0 => S=6
#      τ_b = 6 / sqrt(8*7) ≈ 0.8017837
# LCCS: gold [A,B,C,D,E], pred [A,B,C,D,E] => L=5, P=R=F1=1
# PED:  C: 2→1 (1); D,E: 3→2 (1 each); A,B unchanged
#      MAE = (0+0+1+1+1)/5 = 0.6
# """
#     ),

#     # 7) Slight mis-group: move E up one bucket (keeps order otherwise)
#     dict(
#         name="X7_move_E_up_one",
#         gold=[["A","B"], ["C"], ["D","E"]],
#         pred=[["A","B"], ["C","E"], ["D"]],
#         note="""
# τ-b: n0=10; gold ties: (AB, DE) => n1=2; pred ties: (AB) only => n2=1
#      Concordant: A/B vs all (6) + C vs D (1) => C=7, D=0 => S=7
#      τ_b = 7 / sqrt((10-2)*(10-1)) = 7 / √72 ≈ 0.825
# LCCS: gold [A,B,C,D,E], pred [A,B,C,E,D] => best contiguous match [A,B,C] => L=3
#      P=R=3/5=0.6, F1=0.6
# PED:  E: 3→2 (1); others unchanged => MAE=(0+0+0+0+1)/5 = 0.2
# """
#     ),

#     # 8) Cross misplacements: B pushed to last bucket; D up to middle (both sides change)
#     dict(
#         name="X8_cross_moves",
#         gold=[["A","B"], ["C"], ["D","E"]],
#         pred=[["A"], ["C","D"], ["B","E"]],
#         note="""
# τ-b: n0=10; gold ties: (AB, DE) => n1=2; pred ties: (CD) and (BE) => n2=2
#      Pair-wise:
#        - A vs C/D/E: A precedes all in both => 3 concordant
#        - B vs C/D: B precedes (gold), but follows in pred (group3 vs group2) => 2 discordant
#        - B vs E: tie in pred & gold? (gold: B vs E is B precedes E; pred: tie (BE)) => tie in pred only
#        - C vs E: C precedes E in both => 1 concordant
#        - A vs B: tie gold only; C vs D: tie pred only; D vs E: tie gold only
#      Totals: C=4, D=2 => S=2
#      τ_b = 2 / sqrt((10-2)*(10-2)) = 2/8 = 0.25
# LCCS: gold [A,B,C,D,E], pred [A,C,D,B,E] => best contiguous match [C,D] => L=2
#      P=R=2/5=0.4, F1=0.4
# PED:  A:1→1 (0); B:1→3 (2); C:2→2 (0); D:3→2 (1); E:3→3 (0)
#      MAE = (0+2+0+1+0)/5 = 0.6
# """
#     ),
# ]

# # ---------- Runner ----------
# rows = []
# for t in tests:
#     gold = t["gold"]
#     pred = t["pred"]

#     # LCCS
#     L, P, R, F1, _ = group_aware_lccs(gold, pred, normalize=None, return_alignment=False)
#     # τ-b (extract from ord+cov)
#     oc = evaluate_order_and_coverage(gold, pred, normalize=None, combine=None)
#     tau = get_tau_b(oc)
#     # PED
#     v0, v1, _ = position_error_distance(gold, pred, normalize=None, missing_penalty="max", return_details=False)

#     rows.append({
#         "test": t["name"],
#         "lccs_L": L, "lccs_P": P, "lccs_R": R, "lccs_F1": F1,
#         "tau_b": tau,
#         "ped_v0": v0, "ped_v1": v1,
#         "gold": gold, "pred": pred,
#         "ideal_calc": t["note"].strip()
#     })

# df = pd.DataFrame(rows, columns=[
#     "test","lccs_L","lccs_P","lccs_R","lccs_F1","tau_b","ped_v0","ped_v1","gold","pred","ideal_calc"
# ])
# df


## evaluation pipeline

In [42]:
# ---------------- Normalization ----------------
def _normalize_default(x: Any) -> str:
    return str(x).strip().lower()

# ---------------- Coercion to buckets ----------------

def _coerce_to_buckets(raw: Any, normalize: Optional[Callable[[Any], Any]] = None) -> List[List[Any]]:
    """
    Convert model outputs into List[List[item]] buckets.
    Handles:
      - list[list[str]]
      - list[str] (single bucket)
      - list[dict] (each dict is a bucket: keys are symptoms; ignore values that are "none"/empty)
      - dict with keys among {'groups','buckets','sequence','timeline','temporal_groups',
                              'ordered_groups','result','temporal_result','output'}
      - JSON string or Python-literal string of any of the above
      - Plain text 'A,B | C | D,E'
    """
    norm = (lambda x: x) if normalize is None else normalize

    def _dedupe_bucket(b: Iterable[Any]) -> List[Any]:
        seen, out = set(), []
        for x in b:
            k = norm(x)
            if k in seen or k == "":
                continue
            seen.add(k)
            out.append(k)
        return out

    def _is_none_val(v: Any) -> bool:
        if v is None: return True
        if isinstance(v, str) and norm(v) == "none": return True
        if isinstance(v, (list, tuple, set, dict)) and len(v) == 0: return True
        return False

    # ---------- STRING INPUT: try JSON, then Python literal, then fallback ----------
    if isinstance(raw, str):
        s = raw.strip()

        # 1) JSON attempt
        if s.startswith("{") or s.startswith("[") or s.startswith("```"):
            try:
                obj = json.loads(s.strip("`"))
                return _coerce_to_buckets(obj, normalize=normalize)
            except Exception:
                # 2) Python literal (handles single quotes)
                try:
                    obj = ast.literal_eval(s)
                    return _coerce_to_buckets(obj, normalize=normalize)
                except Exception:
                    pass

        # 3) Fallback: 'A,B | C | D,E' style
        groups = re.split(r"\|\||\||\n+", s)
        buckets = []
        for g in groups:
            toks = [t.strip() for t in re.split(r"[,\t;]+", g) if t.strip()]
            bk = _dedupe_bucket(toks)
            if bk:
                buckets.append(bk)
        return buckets

    # ---------- PYTHON OBJECTS ----------
    if isinstance(raw, (list, tuple)):
        raw = list(raw)
        if len(raw) == 0:
            return []

        # list of dicts -> each dict is a bucket; keep keys whose values are not "none"/empty
        if all(isinstance(g, dict) for g in raw):
            out = []
            for bucket_obj in raw:
                items = [k for k, v in bucket_obj.items() if not _is_none_val(v)]
                bk = _dedupe_bucket(items)
                if bk:
                    out.append(bk)
            return out

        # list of lists/sets
        if all(isinstance(g, (list, tuple, set)) for g in raw):
            return [_dedupe_bucket(list(g)) for g in raw if len(_dedupe_bucket(list(g))) > 0]

        # list of scalars -> single bucket
        if all(not isinstance(g, (list, tuple, set, dict)) for g in raw):
            b = _dedupe_bucket(raw)
            return [b] if b else []

        # mixed: coerce element-wise
        out = []
        for g in raw:
            if isinstance(g, dict):
                items = [k for k, v in g.items() if not _is_none_val(v)]
                bk = _dedupe_bucket(items)
            elif isinstance(g, (list, tuple, set)):
                bk = _dedupe_bucket(list(g))
            else:
                bk = _dedupe_bucket([g])
            if bk:
                out.append(bk)
        return out

    if isinstance(raw, dict):
        # envelopes
        for key in ["groups","buckets","sequence","timeline","temporal_groups",
                    "ordered_groups","result","temporal_result","output"]:
            if key in raw:
                return _coerce_to_buckets(raw[key], normalize=normalize)
        # dict of symptom->mentions (single bucket)
        items = [k for k, v in raw.items() if not _is_none_val(v)]
        b = _dedupe_bucket(items)
        return [b] if b else []

    # Last resort
    return []


def evaluate_temporal_sample(
    gold_raw: Any,
    pred_raw: Any,
    normalize: Optional[Callable[[Any], Any]] = _normalize_default,
    compute_union_position: bool = True,  # also compute order+coverage position score
    lccs_alignment: bool = True,
    combine_method: Optional[str] = "product",  # None | "product" | "geom" | "weighted_geom"
) -> Dict[str, Any]:
    """
    Parse gold/pred, then compute:
      - Exact Match
      - Item coverage P/R/F1
      - Kendall tau-b (ties) on intersection
      - Position Error (intersection) and (optional) union
      - LCCS (len, Rec_G, Prec_P, F1, alignment)
      - Combined = tau_b x item_F1 (if combine_method is not None)
    """
    gold = _coerce_to_buckets(gold_raw, normalize=normalize)
    pred = _coerce_to_buckets(pred_raw, normalize=normalize)

    # print("Coerced Gold:", gold)
    # print("Coerced Pred:", pred)
    # Exact match
    em_ok, em_reason = strict_exact_match(gold, pred, normalize=normalize)

    # Coverage (items only)
    item_P, item_R, item_F1, cov_details = item_set_prf1(gold, pred, normalize=normalize)

    # Kendall tau-b (ties)
    tau_b, tau_details = kendall_tau_b_ties(gold, pred, normalize=normalize, return_details=True)

    # Position error (intersection-only)
    pos_score_int, pos_mae_norm_int, pos_details_int = position_error_distance(
        gold, pred, normalize=normalize, missing_penalty="ignore", return_details=True
    )
    # Position error (union / order+coverage)
    pos_score_union, pos_mae_norm_union, pos_details_union = None, None, None
    if compute_union_position:
        pos_score_union, pos_mae_norm_union, pos_details_union = position_error_distance(
            gold, pred, normalize=normalize, missing_penalty="max", return_details=True
        )

    # LCCS (contiguous)
    lccs_len, lccs_rec_g, lccs_prec_p, lccs_f1, lccs_align = group_aware_lccs(
        gold, pred, normalize=normalize, return_alignment=lccs_alignment
    )

    # Combined (order x coverage)
    combined = None
    if combine_method is not None:
        # Simple product by default: tau_b * item_F1
        if combine_method == "product":
            combined = tau_b * item_F1
        elif combine_method == "geom":
            combined = (tau_b * item_F1) ** 0.5
        elif combine_method == "weighted_geom":
            # default weights tau:0.6, f1:0.4
            w_tau, w_f1 = 0.6, 0.4
            denom = max(1e-9, w_tau + w_f1)
            combined = ((tau_b ** w_tau) * (item_F1 ** w_f1)) ** (1.0 / denom)
        else:
            raise ValueError("combine_method must be one of: None | 'product' | 'geom' | 'weighted_geom'")

    return {
        "exact_match": {"match": em_ok, "reason": em_reason},
        "coverage": {"precision": item_P, "recall": item_R, "f1": item_F1, "details": cov_details},
        "kendall_tau_b": {"tau_b": tau_b, "details": tau_details},
        "position_error_intersection": {
            "score": pos_score_int, "mae_norm": pos_mae_norm_int, "details": pos_details_int
        },
        "position_error_union": None if pos_score_union is None else {
            "score": pos_score_union, "mae_norm": pos_mae_norm_union, "details": pos_details_union
        },
        "lccs": {
            "length": lccs_len, "recall_gold": lccs_rec_g, "precision_pred": lccs_prec_p,
            "f1": lccs_f1, "alignment": lccs_align
        },
        "combined_order_x_coverage": {"method": combine_method, "score": combined},
        "canonical_inputs": {"gold": gold, "pred": pred},  # after coercion/normalization
    }


# pred_example = results_df.final_result_cleaned.to_list()[0]
# gold_example = sample.llm_result_formatted.to_list()[0]

# # print(pred_example)
# # print(gold_example)

# res = evaluate_temporal_sample(
#     gold_example, pred_example,
#     normalize=lambda s: str(s).strip().lower(),
#     compute_union_position=True,
#     lccs_alignment=True,
#     combine_method="product",  # tau_b * item_F1
# )
# # from pprint import pprint
# # pprint(res)

In [43]:
from typing import Any, Callable, Dict, List, Optional, Tuple

# Assumes these exist from earlier messages:
# - evaluate_temporal_sample(gold_raw, pred_raw, normalize=..., compute_union_position=True, lccs_alignment=True, combine_method="product")

def mean_safe(vals: List[Optional[float]]) -> Optional[float]:
    nums = [v for v in vals if v is not None]
    return sum(nums)/len(nums) if nums else None

def evaluate_temporal_batch(
    gold_list: List[Any],
    pred_list: List[Any],
    normalize: Optional[Callable[[Any], Any]] = lambda s: str(s).strip().lower(),
    compute_union_position: bool = True,
    lccs_alignment: bool = False,     # turn on if you want alignments per sample
    combine_method: Optional[str] = "product",  # None | "product" | "geom" | "weighted_geom"
) -> Dict[str, Any]:
    """
    Evaluate multiple (gold, pred) pairs.
    Returns per-sample metrics and macro/micro averages across all successfully evaluated samples.
    """
    if len(gold_list) != len(pred_list):
        raise ValueError(f"gold_list and pred_list must have same length; got {len(gold_list)} vs {len(pred_list)}")

    per_sample: List[Dict[str, Any]] = []
    n = len(gold_list)

    # Micro-coverage accumulators
    micro_gold_total = 0
    micro_pred_total = 0
    micro_inter_total = 0

    # For macro means
    ems, tau_bs = [], []
    cov_Ps, cov_Rs, cov_F1s = [], [], []
    pos_int_scores, pos_int_maes = [], []
    pos_union_scores, pos_union_maes = [], []
    lccs_lens, lccs_recalls, lccs_precs, lccs_f1s = [], [],[], []
    combined_scores = []

    failures = 0
    # counter = 2

    for idx, (g_raw, p_raw) in enumerate(zip(gold_list, pred_list)):
        try:
            res = evaluate_temporal_sample(
                g_raw, p_raw,
                normalize=normalize,
                compute_union_position=compute_union_position,
                lccs_alignment=lccs_alignment,
                combine_method=combine_method,
            )
            per_sample.append({"index": idx, **res})

            # Exact match rate
            ems.append(1.0 if res["exact_match"]["match"] else 0.0)
            # print('ems: ',ems)
            # Coverage (macro)
            cov = res["coverage"]
            cov_Ps.append(cov["precision"])
            cov_Rs.append(cov["recall"])
            cov_F1s.append(cov["f1"])
            # print('coverage: ',cov_F1s)
            # Coverage (micro)
            cd = cov["details"]
            micro_gold_total += cd["gold_count"]
            micro_pred_total += cd["pred_count"]
            micro_inter_total += cd["intersection"]

            # Kendall tau-b
            tau_bs.append(res["kendall_tau_b"]["tau_b"])
            # print('tau_bs: ',tau_bs)
            # Position error (intersection)
            posi = res["position_error_intersection"]
            pos_int_scores.append(posi["score"])
            pos_int_maes.append(posi["mae_norm"])

            # Position error (union) if present
            if res["position_error_union"] is not None:
                posu = res["position_error_union"]
                pos_union_scores.append(posu["score"])
                pos_union_maes.append(posu["mae_norm"])
                # print('pos_union_scores: ',pos_union_scores)
            # LCCS
            lccs = res["lccs"]
            lccs_lens.append(lccs["length"])
            lccs_recalls.append(lccs["recall_gold"])
            lccs_precs.append(lccs["precision_pred"])
            lccs_f1s.append(lccs["f1"])
            # print('lccs_f1s: ',lccs_f1s)
            
            # Combined
            if res["combined_order_x_coverage"]["score"] is not None:
                combined_scores.append(res["combined_order_x_coverage"]["score"])
            
        except Exception as e:
            failures += 1
            per_sample.append({
                "index": idx,
                "error": str(e),
                "raw_gold": g_raw,
                "raw_pred": p_raw,
            })
            # skip from averages

    evaluated = n - failures

    # Macro averages
    macro = {
        "exact_match_rate": mean_safe(ems),
        "coverage_macro": {
            "precision": mean_safe(cov_Ps),
            "recall":    mean_safe(cov_Rs),
            "f1":        mean_safe(cov_F1s),
        },
        "kendall_tau_b": mean_safe(tau_bs),
        "position_error_intersection": {
            "score":   mean_safe(pos_int_scores),
            "mae_norm":mean_safe(pos_int_maes),
        },
        "position_error_union": None if not pos_union_scores else {
            "score":   mean_safe(pos_union_scores),
            "mae_norm":mean_safe(pos_union_maes),
        },
        "lccs": {
            "length":          mean_safe(lccs_lens),
            "recall_gold":     mean_safe(lccs_recalls),
            "precision_pred":  mean_safe(lccs_precs),
            "f1":              mean_safe(lccs_f1s),
        },
        "combined_order_x_coverage": mean_safe(combined_scores) if combined_scores else None,
    }

    # Micro coverage (sums → single P/R/F1)
    if micro_pred_total == 0 and micro_gold_total == 0:
        micro_cov = {"precision": None, "recall": None, "f1": None,
                     "totals": {"gold": 0, "pred": 0, "intersection": 0}}
    else:
        micro_P = (micro_inter_total / micro_pred_total) if micro_pred_total else 0.0
        micro_R = (micro_inter_total / micro_gold_total) if micro_gold_total else 0.0
        micro_F1 = (2 * micro_P * micro_R / (micro_P + micro_R)) if (micro_P + micro_R) else 0.0
        micro_cov = {
            "precision": micro_P,
            "recall": micro_R,
            "f1": micro_F1,
            "totals": {"gold": micro_gold_total, "pred": micro_pred_total, "intersection": micro_inter_total}
        }

    return {
        "n_samples": n,
        "n_evaluated": evaluated,
        "n_failed": failures,
        "macro_averages": macro,
        "micro_coverage": micro_cov,
        "per_sample": per_sample,
    }




In [44]:
# Each entry can be list-of-lists, JSON, dicts with 'groups'/'buckets', or simple "A,B | C | D,E" text.
pred_example = results_df.final_result_cleaned.to_list()
gold_example = sample.llm_result_formatted.to_list()

summary = evaluate_temporal_batch(
    gold_example, pred_example,
    normalize=lambda s: str(s).strip().lower(),
    compute_union_position=True,
    lccs_alignment=False,
    combine_method="product",
)

from pprint import pprint
pprint(summary)

NameError: name 'results_df' is not defined